# Agent 4 — Sleep TST prediction (clean research notebook)

Historical run: 15 November 2026; results are archived in `reports/sleep/`. This is a reproducibility workflow, not an executed notebook or an approval record. Default setup validates only; the development approval cell intentionally stops until a human reviews the private input and enables it. Each subsequent protected stage has its own explicit approval. Use a private working directory; never commit outputs, actual approval records or model binaries.

The historical winner was CatBoost E5, 68 predictors. This notebook refuses to silently replace that choice if a new development run selects a different model. The optional midpoint task is skipped.

## 1. Code and private input mounts

Set paths to your private mounted code and unchanged Agent 2 deliverable. No input slug is guessed and no package installs occur. Prepare pinned offline dependencies from requirements-sleep.txt; verify GPU build/runtime separately.

In [ ]:
import os, sys
from pathlib import Path

# Run from the repository root, or set PMDATA_CODE_ROOT explicitly.
CODE_ROOT = Path(os.environ.get("PMDATA_CODE_ROOT", str(Path.cwd()))).resolve()
if not (CODE_ROOT / "ml/sleep/config.py").is_file() and (CODE_ROOT.parent / "ml/sleep/config.py").is_file():
    CODE_ROOT = CODE_ROOT.parent
INPUT_ROOT = Path(os.environ.get("PMDATA_INPUT_ROOT", str(CODE_ROOT))).resolve()
WORK_ROOT = Path(os.environ.get("PMDATA_WORK_ROOT", "/kaggle/working" if Path("/kaggle/working").is_dir() else str(CODE_ROOT / "private_runs"))).resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SCHEMA_PATH = Path(os.environ.get("PMDATA_SCHEMA_PATH", str(INPUT_ROOT / "research/pmdata/feature_schema.csv"))).resolve()
if not SCHEMA_PATH.is_file():
    SCHEMA_PATH = INPUT_ROOT / "research/feature_schema.csv"
assert (CODE_ROOT / "ml/sleep/config.py").is_file(), "Set PMDATA_CODE_ROOT"
sys.path.insert(0, str(CODE_ROOT))
from ml.sleep import models as sleep_models
assert Path(sleep_models.__file__).resolve().is_relative_to(CODE_ROOT)
assert "catboost_best is not None" in Path(sleep_models.__file__).read_text()
print("Source:", CODE_ROOT, "\nPrivate inputs:", INPUT_ROOT, "\nPrivate outputs:", WORK_ROOT)

In [ ]:

required_files = [
    INPUT_ROOT / "data/processed/sleep_dataset.parquet",
    INPUT_ROOT / "data/processed/shared_features.parquet",
    INPUT_ROOT / "data/processed/split_manifest.csv",
    INPUT_ROOT / "data/processed/grouped_split_manifest.csv",
    INPUT_ROOT / "data/processed/preprocessing_run.json",
    SCHEMA_PATH,
    INPUT_ROOT / "reports/preprocessing_report.md",
    INPUT_ROOT / "reports/leakage_test_report.md",
    INPUT_ROOT / "reports/delivery_integrity.json",
]

for path in required_files:
    status = "OK" if path.is_file() else "MISSING"
    print(f"[{status}] {path.relative_to(INPUT_ROOT)}")


## 2. Explicit configuration

Default mode is `validate`. No fitting or protected evaluation is authorized. The next approval cell switches to development only after an explicit human review.

In [ ]:

from ml.sleep.config import Config

CONFIG = Config(
    # ========================================================
    # Input and output paths
    # ========================================================
    data_dir=str(INPUT_ROOT / "data/processed"),
    schema_path=str(SCHEMA_PATH),
    reports_root=str(INPUT_ROOT / "reports"),
    output_root=str(WORK_ROOT / "sleep_outputs"),

    # ========================================================
    # Execution configuration
    # ========================================================
    mode="validate",
    device="auto",
    seed=42,
    allow_cpu_fallback=True,

    # ========================================================
    # Experiment and tuning configuration
    # ========================================================
    fast_mode=False,
    trials=30,
    fast_trials=5,
    top_k=3,
    early_stopping_rounds=30,
    cpu_threads=2,

    # ========================================================
    # Development stages
    # ========================================================
    run_baselines=False,
    run_feature_experiments=False,
    run_tuning=False,
    run_explainability=False,

    # Sensitivity analysis will be authorized separately
    run_sensitivity_validation=False,

    # ========================================================
    # Protected stages — keep disabled
    # ========================================================
    run_final_test=False,
    run_grouped_outer=False,
    run_sensitivity_test=False,
    run_midpoint=False,
    run_export=False,

    # GPU smoke test has already passed
    run_gpu_smoke=False,

    primary_tst_complete=False,

    # ========================================================
    # Approval and freeze records
    # ========================================================
    approval_record=None,
    frozen_config=None,
    resume_dir=None,
)

print("=" * 65)
print("AGENT 4 DEVELOPMENT CONFIGURATION")
print("=" * 65)

print("Mode:", CONFIG.mode)
print("Device:", CONFIG.device)
print("Fast mode:", CONFIG.fast_mode)
print("Random seed:", CONFIG.seed)

print("\nEnabled stages:")
print("Baselines:", CONFIG.run_baselines)
print("Feature experiments:", CONFIG.run_feature_experiments)
print("Hyperparameter tuning:", CONFIG.run_tuning)
print("Explainability:", CONFIG.run_explainability)

print("\nExperiment budget:")
print("Models:", CONFIG.model_names)
print("Feature experiments:", CONFIG.experiment_ids)

print(
    "Initial configurations:",
    len(CONFIG.model_names) * len(CONFIG.experiment_ids)
)

print(
    "Maximum tuning trials:",
    CONFIG.top_k * CONFIG.trials
)

print("\nProtected stages:")
print("Final test:", CONFIG.run_final_test)
print("Grouped evaluation:", CONFIG.run_grouped_outer)
print("Model export:", CONFIG.run_export)

print("\nConfig fingerprint:", CONFIG.config_fingerprint)
print("Approval record:", CONFIG.approval_record)

print("=" * 65)


## 3. Environment and accelerator inventory

Inventory is not a fit/predict GPU verification. Driver CUDA compatibility differs from installed toolkit/runtime. An explicit synthetic capability flag can later test each installed booster with visible attempt logs; no performance claims follow from fixtures.

In [ ]:
import json
from ml.sleep.logging_utils import environment
from ml.sleep.models import gpu_inventory
print(json.dumps(environment(), indent=2))
print(json.dumps(gpu_inventory(), indent=2))


## 4. Contract and approval gate

Validate unchanged Agent 2 hashes, feature schema, task/shared equality, masks, core sleep, future target metadata and temporal/split/group integrity. Only row counts and gate statuses are displayed; protected target distributions are never shown. Current conditional GO is not human training approval.

In [ ]:
from ml.sleep.data_validation import load_bundle
BUNDLE = load_bundle(CONFIG)
print(json.dumps(BUNDLE.report, indent=2))


In [ ]:

import json

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace

from ml.sleep.config import require_approval

# ============================================================
# 1. Verify development configuration and input validation
# ============================================================

assert CONFIG.mode in ("validate", "develop")

assert BUNDLE.report["status"] == "software_contract_valid"

assert all(BUNDLE.report["checks"].values()), (
    "Input validation checks have not all passed"
)

assert not CONFIG.run_final_test
assert not CONFIG.run_grouped_outer
assert not CONFIG.run_export

# ============================================================
# 2. Human approval
# ============================================================

# Before setting this to True, the ML lead must verify:
#
# - Preprocessing validation and leakage reports.
# - Audit count reconciliation.
# - Timestamp and ingestion limitations.
# - Shared feature contract.
# - Approved preprocessing version merged to main.
# - Private, authorized PMData research usage.

ML_LEAD_APPROVED = False

APPROVED_BY = os.environ.get("PMDATA_APPROVER", "").strip()

if not ML_LEAD_APPROVED:
    raise PermissionError(
        "Review the preprocessing evidence, then set "
        "ML_LEAD_APPROVED=True to authorize development."
    )

# ============================================================
# 3. Enable development only after explicit review
# ============================================================
assert APPROVED_BY, "Set PMDATA_APPROVER to the reviewing ML lead"
CONFIG = replace(CONFIG, mode="develop", run_baselines=True, run_feature_experiments=True, run_tuning=True, run_explainability=True)

# 3. Load original Agent 4 approval template
# ============================================================

template_path = (
    CODE_ROOT / "configs/sleep/approval_record.template.json"
)

approval = json.loads(
    template_path.read_text(encoding="utf-8")
)

# ============================================================
# 4. Bind approval to the actual development configuration
# ============================================================

approval.update({
    "approved": True,

    "approved_by": APPROVED_BY,

    "approved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "task": "sleep",

    "gate_status": "CONDITIONAL GO",

    "preprocessing_reviewed": True,

    "preprocessing_merged_to_main": True,

    "preprocessing_version": BUNDLE.metadata["version"],

    "preprocessing_fingerprint": (
        BUNDLE.fingerprints["preprocessing_fingerprint"]
    ),

    "schema_fingerprint": (
        BUNDLE.fingerprints["schema_fingerprint"]
    ),

    "split_fingerprint": (
        BUNDLE.fingerprints["split_fingerprint"]
    ),

    "dataset_fingerprint": (
        BUNDLE.fingerprints["dataset_fingerprint"]
    ),

    "config_fingerprint": CONFIG.config_fingerprint,

    "reviewed_evidence": {
        "mandatory_validation": True,
        "audit_reconciliation": True,
        "timestamp_assumptions": True,
        "shared_contract": True
    },

    # Development only — protected stages remain locked
    "authorized_scopes": [
        "develop"
    ],

    "approved_freeze_fingerprint": None,

    "scenario_fingerprints": {},

    "primary_tst_workflow_complete": False,

    "note": (
        "Authorized for chronological train/validation "
        "development only. No protected test evaluation, "
        "grouped outer evaluation, or model export. "
        "Unknown participant timezones and ingestion "
        "assumptions remain documented limitations."
    )
})

# ============================================================
# 5. Save the approval record
# ============================================================

APPROVAL_PATH = WORK_ROOT / "sleep_approval_development.json"

APPROVAL_PATH.write_text(
    json.dumps(approval, indent=2),
    encoding="utf-8"
)

# ============================================================
# 6. Attach approval to CONFIG
# ============================================================

CONFIG = replace(
    CONFIG,
    approval_record=str(APPROVAL_PATH)
)

# ============================================================
# 7. Verify using the original Agent 4 approval gate
# ============================================================

verified_approval = require_approval(
    CONFIG,
    BUNDLE,
    "develop"
)

print("=" * 65)
print("DEVELOPMENT APPROVAL VERIFIED")
print("=" * 65)

print("Approved by:", verified_approval["approved_by"])
print("Gate status:", verified_approval["gate_status"])
print("Authorized scopes:", verified_approval["authorized_scopes"])

print("\nApproval file:", APPROVAL_PATH)

print("Config fingerprint:", CONFIG.config_fingerprint)

print("\nDevelopment authorized: True")
print("Final test authorized: False")
print("Grouped evaluation authorized: False")
print("Model export authorized: False")

print("=" * 65)


## 5. Authorized orchestration and live progress

The reusable runner measures actual UTC and perf-counter times, logs every baseline/model/trial/fold/failure/fallback, and persists results immediately. Default execution below creates a validation-only run and prints blocked scientific stages. For an authorized future run, use the configuration and approval/freeze templates documented in README.md. No training code is duplicated here.

In [ ]:

from ml.sleep.train import run
from datetime import datetime, timezone
from time import perf_counter

# ============================================================
# Verify development configuration
# ============================================================

assert CONFIG.mode == "develop"
assert CONFIG.approval_record is not None

assert CONFIG.run_baselines
assert CONFIG.run_feature_experiments
assert CONFIG.run_tuning

assert not CONFIG.run_final_test
assert not CONFIG.run_grouped_outer
assert not CONFIG.run_export
assert not CONFIG.run_gpu_smoke

# ============================================================
# Start actual PMData training
# ============================================================

print("=" * 65)
print("AGENT 4 — SLEEP PREDICTION DEVELOPMENT")
print("=" * 65)

print("Mode:", CONFIG.mode)
print("Device:", CONFIG.device)
print("Fast mode:", CONFIG.fast_mode)
print("Seed:", CONFIG.seed)

print("\nPlanned experiments:")
print("- 4 baseline experiments")
print("- 30 model-feature configurations")
print("- Up to 90 hyperparameter tuning trials")
print("- Validation explainability")

print("\nConfig fingerprint:", CONFIG.config_fingerprint)

print(
    "\nTraining started at:",
    datetime.now(timezone.utc).isoformat()
)

print("=" * 65, flush=True)

start_time = perf_counter()

# Run the authorized Agent 4 development workflow
RUN_DIR = run(CONFIG)

# ============================================================
# Final runtime summary
# ============================================================

elapsed = perf_counter() - start_time

print("\n" + "=" * 65)
print("DEVELOPMENT WORKFLOW COMPLETED")
print("=" * 65)

print(
    "Finished at:",
    datetime.now(timezone.utc).isoformat()
)

print(f"Total runtime: {elapsed:.2f} seconds")
print(f"Total runtime: {elapsed / 60:.2f} minutes")

print("\nPrivate run directory:", RUN_DIR)

print("=" * 65)


## 6. Baselines

Authorized development runs B0 train median, B1 exact prior sleep, B2 true calendar 7-day median min4 with persistence fallback, B3 personal historical median with fallback. B2 reads the full upstream shared calendar grid, never sparse target rows or the rolling mean. Coverage and construction/score timings are logged.

## 7. E0–E5 model comparison

Five required regressors × six frozen feature groups = 30 configurations. Rows/splits remain identical. Median imputation/scaling is training-only; all-null training columns retain schema using a deterministic constant. Optional null modalities never remove rows. Initial model/feature leaderboard uses validation MAE and participant-macro MAE tie-break.

## 8. Hyperparameter tuning and selection

Seeded Optuna TPE tunes top 3 configurations, 30 trials (FAST 5); one GPU trial at a time. SQLite, per-trial params/effective iterations/timing and study fingerprints are saved. Full matching identities are required for resume. No held-out test/group outcome enters the objective. Selected configuration is a proposal until human freeze.

## 9. Validation dashboards

Only existing authorized-stage results are shown. No placeholder metrics are fabricated.

In [ ]:

import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

METRICS_DIR = Path(RUN_DIR) / "metrics"

def show_leaderboard(filename, title):
    path = METRICS_DIR / filename

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    if not path.is_file():
        print(f"File not found: {path}")
        return None

    df = pd.read_csv(path)

    # Identify the validation MAE column
    possible_columns = [
        "validation_mae",
        "validation_mae_minutes",
        "val_mae",
        "mae_minutes",
        "mae_min",
        "mae"
    ]

    mae_col = next(
        (col for col in possible_columns if col in df.columns),
        None
    )

    if mae_col:
        df = df.sort_values(mae_col).reset_index(drop=True)
        df.insert(0, "rank", range(1, len(df) + 1))

    display(df)

    return df


# ============================================================
# 1. BASELINE LEADERBOARD
# ============================================================

baseline_results = show_leaderboard(
    "baseline_metrics_validation.csv",
    "BASELINE LEADERBOARD — B0 TO B3"
)


# ============================================================
# 2. INITIAL MODEL LEADERBOARD
# ============================================================

initial_results = show_leaderboard(
    "model_feature_results.csv",
    "INITIAL MODEL LEADERBOARD — E0 TO E5"
)


# ============================================================
# 3. HYPERPARAMETER TUNING RESULTS
# ============================================================

print("\n" + "=" * 80)
print("HYPERPARAMETER TUNING RESULTS")
print("=" * 80)

tuning_files = [
    path for path in METRICS_DIR.glob("*.csv")
    if any(
        keyword in path.name.lower()
        for keyword in ["tuning", "tuned", "optuna"]
    )
]

if tuning_files:
    for path in sorted(tuning_files):
        show_leaderboard(path.name, path.stem.upper())
else:
    print("No tuning CSV found in the metrics directory.")


# ============================================================
# 4. AVAILABLE RESULT FILES
# ============================================================

print("\n" + "=" * 80)
print("AVAILABLE RESULT FILES")
print("=" * 80)

for path in sorted(Path(RUN_DIR).rglob("*")):
    if path.is_file() and path.suffix in [".csv", ".json"]:
        print(path.relative_to(RUN_DIR))



## 10. Configuration freeze and protected chronological test

The ML lead pins exact model/features/params/iterations/transforms/checkpoint/data/schema/split/config/protocol and freeze fingerprint. `mode=final`, `run_final_test=True`, explicit final_execution/final_test scopes and matching human approval are all required. Train-fitted model weights remain frozen. Test outcomes never change the selected model.

In [ ]:

# ============================================================
# AGENT 4 — MODEL FREEZE REVIEW
# No training and no protected test evaluation
# ============================================================

import json
import joblib
import numpy as np

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace

from ml.sleep.config import (
    require_approval,
    require_freeze,
    fingerprint,
    file_hash
)

from ml.sleep.features import select_features
from ml.sleep.logging_utils import dependency_environment
from ml.sleep.train import code_fingerprint

# ============================================================
# 1. Verify original development run
# ============================================================

assert CONFIG.mode == "develop"

assert not CONFIG.run_final_test
assert not CONFIG.run_grouped_outer
assert not CONFIG.run_export

DEVELOPMENT_APPROVAL = require_approval(
    CONFIG,
    BUNDLE,
    "develop"
)

DEV_RUN_DIR = Path(RUN_DIR)

PROPOSAL_PATH = (
    DEV_RUN_DIR
    / "metadata"
    / "selection_proposal.json"
)

assert PROPOSAL_PATH.is_file(), (
    "Original development proposal not found"
)

SELECTION = json.loads(
    PROPOSAL_PATH.read_text()
)

RUN_IDENTITY = json.loads(
    (
        DEV_RUN_DIR
        / "metadata"
        / "resume_identity.json"
    ).read_text()
)

# ============================================================
# 2. Verify model selection
# ============================================================

assert SELECTION["model_name"] == "catboost"
assert SELECTION["experiment_id"] == "E5"
assert SELECTION["stage"] == "tuned"

assert SELECTION["fit_policy"] == (
    "train_only_no_validation_refit"
)

assert np.isfinite(SELECTION["mae_minutes"])

assert int(SELECTION["params"]["depth"]) > 0
assert int(SELECTION["params"]["iterations"]) > 0

assert len(SELECTION["ordered_features"]) == 68

assert SELECTION["ordered_features"] == (
    select_features(BUNDLE.schema, "E5")
)

# ============================================================
# 3. Verify original checkpoint integrity
# ============================================================

CHECKPOINT_PATH = Path(
    SELECTION["checkpoint"]
)

assert CHECKPOINT_PATH.is_file(), (
    f"Original checkpoint missing: {CHECKPOINT_PATH}"
)

CHECKPOINT_SHA256 = file_hash(
    CHECKPOINT_PATH
)

assert CHECKPOINT_SHA256 == (
    SELECTION["checkpoint_sha256"]
)



# Load only our trusted, locally generated checkpoint.

FROZEN_MODEL = joblib.load(
    CHECKPOINT_PATH
)

assert FROZEN_MODEL.model_name == "catboost"

assert list(FROZEN_MODEL.columns) == (
    SELECTION["ordered_features"]
)

assert int(
    FROZEN_MODEL.estimator.tree_count_
) == SELECTION["params"]["iterations"]

# ============================================================
# 4. Verify data and environment identity
# ============================================================

for key, value in BUNDLE.fingerprints.items():

    assert RUN_IDENTITY[key] == value, (
        f"Development data identity mismatch: {key}"
    )

assert RUN_IDENTITY["config_fingerprint"] == (
    CONFIG.config_fingerprint
)

CURRENT_ENVIRONMENT_FINGERPRINT = fingerprint(
    dependency_environment()
)

assert RUN_IDENTITY["environment_fingerprint"] == (
    CURRENT_ENVIRONMENT_FINGERPRINT
)

CURRENT_CODE_FINGERPRINT = code_fingerprint()

assert RUN_IDENTITY["code_fingerprint"] == (
    CURRENT_CODE_FINGERPRINT
)

# ============================================================
# 5. Display candidate for human review
# ============================================================

print("=" * 75)
print("MODEL FREEZE REVIEW — ALL CHECKS PASSED")
print("=" * 75)

print("Development run:", DEV_RUN_DIR.name)

print("\nModel:", SELECTION["model_name"])
print("Feature experiment:", SELECTION["experiment_id"])
print("Number of predictors:", len(SELECTION["ordered_features"]))

print("\nHyperparameters:")

print(
    json.dumps(
        SELECTION["params"],
        indent=2
    )
)

print("\nValidation metrics:")

print(
    "MAE:",
    round(SELECTION["mae_minutes"], 6),
    "minutes"
)

print(
    "RMSE:",
    round(SELECTION["rmse_minutes"], 6),
    "minutes"
)

print(
    "Participant-macro MAE:",
    round(
        SELECTION["participant_macro_mae_minutes"],
        6
    ),
    "minutes"
)

print(
    "Improvement vs persistence:",
    round(
        SELECTION["improvement_vs_persistence_pct"],
        4
    ),
    "%"
)

print("\nCheckpoint:", CHECKPOINT_PATH)
print("Checkpoint SHA256:", CHECKPOINT_SHA256)

print("\nData fingerprints:")

for key, value in BUNDLE.fingerprints.items():
    print(key, ":", value)

print(
    "\nConfiguration fingerprint:",
    CONFIG.config_fingerprint
)

print(
    "Environment fingerprint:",
    CURRENT_ENVIRONMENT_FINGERPRINT
)

print(
    "Code fingerprint:",
    CURRENT_CODE_FINGERPRINT
)

print("\nFit policy:", SELECTION["fit_policy"])

print("\nCheckpoint integrity: VERIFIED")
print("Model identity: VERIFIED")
print("Feature order: VERIFIED")
print("Preprocessing identity: VERIFIED")
print("Environment identity: VERIFIED")
print("Code identity: VERIFIED")

print("\nProtected test accessed: NO")
print("Model retrained: NO")
print("Production export performed: NO")

print("=" * 75)



In [ ]:

# ============================================================
# AGENT 4 — OFFICIAL MODEL FREEZE
# Human approval + immutable selection record
# ============================================================

import json

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace

from ml.sleep.config import (
    fingerprint,
    file_hash,
    require_approval,
    require_freeze
)

from ml.sleep.experiments import load_frozen_model

# ============================================================
# 1. Explicit human approval
# ============================================================

# Set to True only after reviewing the previous cell
# and accepting the final model selection.

ML_LEAD_APPROVES_FREEZE = False
if not ML_LEAD_APPROVES_FREEZE:

    raise PermissionError(
        "Review the original CatBoost E5 selection, "
        "then set ML_LEAD_APPROVES_FREEZE=True "
        "to freeze the model and authorize its "
        "chronological final evaluation."
    )

APPROVED_BY = DEVELOPMENT_APPROVAL["approved_by"]

FROZEN_AT = datetime.now(
    timezone.utc
).isoformat()

# ============================================================
# 2. Recheck checkpoint before freezing
# ============================================================

assert file_hash(CHECKPOINT_PATH) == (
    SELECTION["checkpoint_sha256"]
)

assert CONFIG.mode == "develop"

assert not CONFIG.run_final_test
assert not CONFIG.run_grouped_outer
assert not CONFIG.run_export

# ============================================================
# 3. Create dedicated freeze directory
# ============================================================

FREEZE_DIR = (
    DEV_RUN_DIR
    / "frozen_selection"
    / datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )
)

FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=False
)

FREEZE_PATH = (
    FREEZE_DIR
    / "frozen_config.json"
)

FINAL_APPROVAL_PATH = (
    FREEZE_DIR
    / "approval_final_test.json"
)

# ============================================================
# 4. Construct frozen model payload
# ============================================================

FROZEN_PAYLOAD = {
    "approved_by": APPROVED_BY,

    "frozen_at_utc": FROZEN_AT,

    "chronological_selection_frozen": True,

    "config_fingerprint": CONFIG.config_fingerprint,

    **BUNDLE.fingerprints,

    "environment_fingerprint": (
        CURRENT_ENVIRONMENT_FINGERPRINT
    ),

    "code_fingerprint": (
        CURRENT_CODE_FINGERPRINT
    ),

    "authorized_protocols": [
        "final_test"
    ],

    "nested_selection_protocol": (
        "groupkfold_inner_full_matrix_v1"
    ),

    "evaluation_protocol": (
        CONFIG.evaluation_protocol
    ),

    "selection": SELECTION,

    "selection_rationale": {
        "primary_metric": (
            "validation_mae_minutes"
        ),

        "secondary_metric": (
            "participant_macro_mae_minutes"
        ),

        "selected_model": (
            "Original CatBoost E5 Tuned"
        ),

        "reason": (
            "Lowest chronological February validation MAE "
            "among the evaluated ML candidates, with "
            "lower RMSE and participant-macro MAE "
            "than the personalized B3 baseline."
        ),

        "additional_experiments_reviewed": [
            "Personalized residual learning",
            "Targeted feature ablation F1-F6",
            "Advanced tuning F5/F6"
        ],

        "additional_experiments_selected": False,

        "baseline_b3_validation_mae": float(pd.read_csv(DEV_RUN_DIR / "metrics/baseline_metrics_validation.csv").loc[lambda d: d.experiment_id.eq("B3"), "mae_minutes"].iloc[0]),

        "best_advanced_internal_candidate": (
            "T1 F5 RMSE"
        ),

        "validation_reuse_limitation": (
            "February and internal chronological folds "
            "were reused during development. "
            "Final chronological test remains independent "
            "of model selection."
        )
    },

    "limitations": [
        "Small terrestrial wearable dataset",
        "Unknown participant timezones",
        "Unknown ingestion timing",
        "No clinical or astronaut-domain validation",
        "Small validation improvement versus B3"
    ],

    "test_used_for_selection": False,

    "production_export_authorized": False
}

# ============================================================
# 5. Fingerprint the complete frozen payload
# ============================================================

FREEZE_FINGERPRINT = fingerprint(
    FROZEN_PAYLOAD
)

FREEZE_RECORD = {
    "frozen_payload": FROZEN_PAYLOAD,

    "freeze_fingerprint": FREEZE_FINGERPRINT,

    "note": (
        "Human-reviewed original development checkpoint. "
        "No retraining, feature changes or parameter "
        "adjustments after freeze. "
        "Only chronological final test is authorized."
    )
}

FREEZE_PATH.write_text(
    json.dumps(
        FREEZE_RECORD,
        indent=2
    ),
    encoding="utf-8"
)

# ============================================================
# 6. Create separately scoped final approval
# ============================================================

FINAL_APPROVAL = dict(
    DEVELOPMENT_APPROVAL
)

FINAL_APPROVAL.update({
    "approved": True,

    "approved_by": APPROVED_BY,

    "approved_at_utc": FROZEN_AT,

    "authorized_scopes": [
        "develop",
        "final_execution",
        "final_test"
    ],

    "approved_freeze_fingerprint": (
        FREEZE_FINGERPRINT
    ),

    "dataset_fingerprint": (
        BUNDLE.fingerprints["dataset_fingerprint"]
    ),

    "primary_tst_workflow_complete": False,

    "note": (
        "Authorizes one protected chronological final "
        "evaluation of the original frozen CatBoost E5 "
        "checkpoint. No grouped evaluation, sensitivity "
        "test, midpoint evaluation or production export "
        "is authorized by this record."
    )
})

FINAL_APPROVAL_PATH.write_text(
    json.dumps(
        FINAL_APPROVAL,
        indent=2
    ),
    encoding="utf-8"
)

# ============================================================
# 7. Prepare final execution configuration
# ============================================================

FINAL_CONFIG = replace(
    CONFIG,

    mode="final",

    approval_record=str(
        FINAL_APPROVAL_PATH
    ),

    frozen_config=str(
        FREEZE_PATH
    ),

    resume_dir=None,

    run_final_test=True,

    run_grouped_outer=False,

    run_sensitivity_test=False,

    run_export=False
)

# IMPORTANT:
# Preserve all semantic development settings
# so that the configuration fingerprint remains identical.

assert FINAL_CONFIG.config_fingerprint == (
    CONFIG.config_fingerprint
)

# ============================================================
# 8. Verify official Agent 4 freeze gates
# ============================================================

require_approval(
    FINAL_CONFIG,
    BUNDLE,
    "final_execution"
)

VERIFIED_FREEZE = require_freeze(
    FINAL_CONFIG,
    BUNDLE,
    "final_test"
)

# Validate the trained model without accessing test data.

verified_body, verified_model, verified_columns = (
    load_frozen_model(
        FINAL_CONFIG,
        BUNDLE,
        "final_test"
    )
)

assert fingerprint(verified_body) == (
    FREEZE_FINGERPRINT
)

assert verified_columns == (
    SELECTION["ordered_features"]
)

assert verified_model.model_name == (
    SELECTION["model_name"]
)

# ============================================================
# 9. Save final configuration
# ============================================================

FINAL_CONFIG_PATH = (
    FREEZE_DIR
    / "final_execution_config.json"
)

FINAL_CONFIG_PATH.write_text(
    json.dumps(
        FINAL_CONFIG.payload(),
        indent=2
    ),
    encoding="utf-8"
)

# ============================================================
# 10. Display verified freeze summary
# ============================================================

print("=" * 75)
print("OFFICIAL MODEL FREEZE COMPLETED")
print("=" * 75)

print("Approved by:", APPROVED_BY)
print("Frozen at:", FROZEN_AT)

print("\nFrozen model:", SELECTION["model_name"])
print("Feature experiment:", SELECTION["experiment_id"])
print("Number of predictors:", len(verified_columns))

print(
    "Frozen validation MAE:",
    round(SELECTION["mae_minutes"], 6)
)

print("\nCheckpoint:", CHECKPOINT_PATH)

print(
    "Checkpoint SHA256:",
    CHECKPOINT_SHA256
)

print(
    "\nFreeze fingerprint:",
    FREEZE_FINGERPRINT
)

print(
    "Configuration fingerprint:",
    FINAL_CONFIG.config_fingerprint
)

print("\nFrozen configuration:", FREEZE_PATH)

print(
    "Final approval record:",
    FINAL_APPROVAL_PATH
)

print(
    "Final execution config:",
    FINAL_CONFIG_PATH
)

print("\nFrozen protocols:",
      FROZEN_PAYLOAD["authorized_protocols"])

print("\nFreeze verification: PASSED")
print("Checkpoint verification: PASSED")

print("\nFinal chronological test authorized: YES")
print("Grouped evaluation authorized: NO")
print("Sensitivity test authorized: NO")
print("Model export authorized: NO")

print("\nFinal test actually executed: NO")
print("Model retrained: NO")

print("=" * 75)


In [ ]:

# ============================================================
# AGENT 4 — FINAL CHRONOLOGICAL TEST EVALUATION
# Frozen CatBoost E5 | March 2020
# ============================================================

import json

from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter

from ml.sleep.train import run
from ml.sleep.config import (
    require_approval,
    require_freeze,
    file_hash
)

# ============================================================
# 1. Verify authorization and frozen checkpoint
# ============================================================

assert FINAL_CONFIG.mode == "final"

assert FINAL_CONFIG.run_final_test

assert not FINAL_CONFIG.run_grouped_outer
assert not FINAL_CONFIG.run_sensitivity_test
assert not FINAL_CONFIG.run_export
assert not FINAL_CONFIG.run_gpu_smoke

require_approval(
    FINAL_CONFIG,
    BUNDLE,
    "final_execution"
)

require_freeze(
    FINAL_CONFIG,
    BUNDLE,
    "final_test"
)

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256

# Prevent accidental repeated final evaluation.

TEST_STARTED_MARKER = (
    FREEZE_DIR / "final_test_started.json"
)

TEST_COMPLETED_MARKER = (
    FREEZE_DIR / "final_test_completed.json"
)

assert not TEST_STARTED_MARKER.exists(), (
    "Final test has already been initiated. "
    "Inspect the existing run before attempting another."
)

# ============================================================
# 2. Record test execution before accessing outcomes
# ============================================================

started_at = datetime.now(
    timezone.utc
).isoformat()

TEST_STARTED_MARKER.write_text(
    json.dumps(
        {
            "started_at_utc": started_at,
            "freeze_fingerprint": FREEZE_FINGERPRINT,
            "checkpoint_sha256": CHECKPOINT_SHA256,
            "protocol": "final_test",
            "status": "started"
        },
        indent=2
    )
)

print("=" * 75)
print("PROTECTED FINAL CHRONOLOGICAL EVALUATION")
print("=" * 75)

print("Frozen model:", SELECTION["model_name"])
print("Feature configuration:", SELECTION["experiment_id"])
print("Number of predictors:", len(SELECTION["ordered_features"]))

print("\nEvaluation period: March 2020")
print("Expected test samples: 296")

print("\nValidation MAE:", SELECTION["mae_minutes"])
print("Bootstrap replicates:", FINAL_CONFIG.bootstrap_replicates)

print("\nFreeze fingerprint:", FREEZE_FINGERPRINT)

print("\nTraining performed: NO")
print("Hyperparameter changes: NO")
print("Feature changes: NO")

print("\nStarted at:", started_at)
print("=" * 75, flush=True)

# ============================================================
# 3. Execute the authorized final test
# ============================================================

start_time = perf_counter()

FINAL_RUN_DIR = run(FINAL_CONFIG)

elapsed = perf_counter() - start_time

# ============================================================
# 4. Verify final results
# ============================================================

FINAL_RUN_DIR = Path(FINAL_RUN_DIR)

metrics_path = (
    FINAL_RUN_DIR
    / "metrics"
    / "final_test_metrics.json"
)

baseline_path = (
    FINAL_RUN_DIR
    / "metrics"
    / "baseline_metrics_test.csv"
)

assert metrics_path.is_file()
assert baseline_path.is_file()

final_metrics = json.loads(
    metrics_path.read_text()
)

assert final_metrics["n"] == 296

assert final_metrics["freeze_fingerprint"] == (
    FREEZE_FINGERPRINT
)

# Ensure model weights have not changed.

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256

# ============================================================
# 5. Record successful completion
# ============================================================

TEST_COMPLETED_MARKER.write_text(
    json.dumps(
        {
            "status": "completed",
            "started_at_utc": started_at,
            "finished_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "final_run_dir": str(FINAL_RUN_DIR),
            "freeze_fingerprint": FREEZE_FINGERPRINT,
            "checkpoint_sha256": CHECKPOINT_SHA256,
            "runtime_seconds": elapsed
        },
        indent=2
    )
)

print("\n" + "=" * 75)
print("FINAL CHRONOLOGICAL TEST COMPLETED")
print("=" * 75)

print("Test samples:", final_metrics["n"])
print("Participants:", final_metrics["participants"])

print(
    "Test MAE:",
    round(final_metrics["mae_minutes"], 4),
    "minutes"
)

print(
    "Test RMSE:",
    round(final_metrics["rmse_minutes"], 4),
    "minutes"
)

print(
    "Participant-macro MAE:",
    round(
        final_metrics["participant_macro_mae_minutes"],
        4
    ),
    "minutes"
)

print(
    "Improvement vs persistence:",
    final_metrics["improvement_vs_persistence_pct"],
    "%"
)

print("\nCheckpoint integrity: VERIFIED")
print("Model retrained: NO")

print("Total runtime:", round(elapsed, 2), "seconds")

print("\nFinal run directory:", FINAL_RUN_DIR)

print("=" * 75)


In [ ]:

# ============================================================
# FINAL TEST RESULTS DASHBOARD
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display, Image

# ============================================================
# 1. Load final evaluation results
# ============================================================

final_metrics = json.loads(
    (
        FINAL_RUN_DIR
        / "metrics"
        / "final_test_metrics.json"
    ).read_text()
)

test_baselines = pd.read_csv(
    FINAL_RUN_DIR
    / "metrics"
    / "baseline_metrics_test.csv"
)

metric_columns = [
    "mae_minutes",
    "rmse_minutes",
    "participant_macro_mae_minutes",
    "mean_signed_error_minutes",
    "n",
    "participants",
    "improvement_vs_persistence_pct"
]

model_row = {
    "experiment_id": "CatBoost E5 Tuned",
    **{
        key: final_metrics[key]
        for key in metric_columns
    }
}

test_leaderboard = pd.concat(
    [
        test_baselines,
        pd.DataFrame([model_row])
    ],
    ignore_index=True
)

b3_test_mae = test_leaderboard.loc[
    test_leaderboard["experiment_id"] == "B3",
    "mae_minutes"
].iloc[0]

test_leaderboard["improvement_vs_B3_pct"] = (
    (
        b3_test_mae
        - test_leaderboard["mae_minutes"]
    )
    / b3_test_mae
    * 100
)

test_leaderboard = test_leaderboard.sort_values(
    [
        "mae_minutes",
        "participant_macro_mae_minutes"
    ]
).reset_index(drop=True)

test_leaderboard.insert(
    0,
    "rank",
    range(1, len(test_leaderboard) + 1)
)

test_leaderboard.to_csv(
    FINAL_RUN_DIR
    / "metrics"
    / "final_test_leaderboard.csv",
    index=False
)

print("=" * 75)
print("FINAL TEST LEADERBOARD — MARCH 2020")
print("=" * 75)

display(
    test_leaderboard[
        [
            "rank",
            "experiment_id",
            "mae_minutes",
            "rmse_minutes",
            "participant_macro_mae_minutes",
            "improvement_vs_persistence_pct",
            "improvement_vs_B3_pct",
            "n",
            "participants"
        ]
    ].round(4)
)

# ============================================================
# 2. Validation versus test
# ============================================================

generalization_table = pd.DataFrame([
    {
        "metric": metric,

        "validation": SELECTION[metric],

        "test": final_metrics[metric]
    }
    for metric in [
        "mae_minutes",
        "rmse_minutes",
        "participant_macro_mae_minutes"
    ]
])

generalization_table["change_pct"] = (
    (
        generalization_table["test"]
        - generalization_table["validation"]
    )
    / generalization_table["validation"]
    * 100
)

generalization_table.to_csv(
    FINAL_RUN_DIR
    / "metrics"
    / "validation_vs_test.csv",
    index=False
)

print("\nVALIDATION VS TEST — GENERALIZATION")

display(
    generalization_table.round(4)
)

# ============================================================
# 3. Participant-cluster bootstrap confidence intervals
# ============================================================

bootstrap = final_metrics[
    "participant_cluster_bootstrap"
]

interval_rows = []

for metric, interval in (
    bootstrap.get("intervals") or {}
).items():

    if interval is not None:

        interval_rows.append({
            "metric": metric,
            "point_estimate": final_metrics.get(metric),
            "lower_95": interval["lower"],
            "upper_95": interval["upper"],
            "valid_replicates": interval["valid_replicates"]
        })

bootstrap_table = pd.DataFrame(
    interval_rows
)

bootstrap_table.to_csv(
    FINAL_RUN_DIR
    / "metrics"
    / "bootstrap_confidence_intervals.csv",
    index=False
)

print("\nPARTICIPANT-CLUSTER BOOTSTRAP — 95% CI")

display(
    bootstrap_table.round(4)
)

print(
    "\nBootstrap note:",
    bootstrap.get("warning", "")
)

# ============================================================
# 4. Test performance visualization
# ============================================================

plot_data = test_leaderboard.sort_values(
    "mae_minutes",
    ascending=False
)

fig, ax = plt.subplots(figsize=(10, 5))

colors = [
    "#2A9D8F" if name == "CatBoost E5 Tuned"
    else "#E69F00" if name == "B3"
    else "#8C9BAA"
    for name in plot_data["experiment_id"]
]

bars = ax.barh(
    plot_data["experiment_id"],
    plot_data["mae_minutes"],
    color=colors
)

ax.bar_label(
    bars,
    fmt="%.2f",
    padding=5
)

ax.set_xlabel("Test MAE (minutes)")
ax.set_title("Final Sleep Prediction — March 2020")

ax.set_xlim(
    0,
    plot_data["mae_minutes"].max() * 1.18
)

ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

test_chart_path = (
    FINAL_RUN_DIR
    / "figures"
    / "final_test_leaderboard.png"
)

fig.savefig(
    test_chart_path,
    dpi=160,
    bbox_inches="tight"
)

plt.close(fig)

display(
    Image(filename=str(test_chart_path))
)

# ============================================================
# 5. Original prediction diagnostics
# ============================================================

diagnostics_path = (
    FINAL_RUN_DIR
    / "figures"
    / "protected_test_error_analysis.png"
)

if diagnostics_path.is_file():

    print("\nFINAL TEST PREDICTION DIAGNOSTICS")

    display(
        Image(filename=str(diagnostics_path))
    )


## 11. Participant-independent nested grouped evaluation

`run_grouped_outer=False` by default. Once chronology is frozen and separately authorized, each saved outer fold selects/tunes entirely within outer-training participants using inner GroupKFold, then evaluates the held-out subjects once. It does not recycle a global chronological winner as independent tuning. Strictly past personal history may initialize baseline context; no population fitting uses held-out participants.

In [ ]:

# ============================================================
# AGENT 4 — GROUPED EVALUATION AUTHORIZATION
# ============================================================

import json

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace

from ml.sleep.config import (
    fingerprint,
    file_hash,
    require_approval,
    require_freeze
)

from ml.sleep.train import code_fingerprint

# ============================================================
# 1. Verify completed chronological evaluation
# ============================================================

assert TEST_COMPLETED_MARKER.is_file()

completion = json.loads(
    TEST_COMPLETED_MARKER.read_text()
)

assert completion["status"] == "completed"

assert completion["freeze_fingerprint"] == (
    FREEZE_FINGERPRINT
)

assert file_hash(CHECKPOINT_PATH) == (
    CHECKPOINT_SHA256
)

# ============================================================
# 2. Explicit human authorization
# ============================================================

ML_LEAD_APPROVES_GROUPED = False
if not ML_LEAD_APPROVES_GROUPED:

    raise PermissionError(
        "Review the chronological test results "
        "and authorize participant-independent "
        "nested grouped evaluation."
    )

# ============================================================
# 3. Preserve original frozen selection
# ============================================================

original_freeze = json.loads(
    FREEZE_PATH.read_text()
)

original_body = original_freeze["frozen_payload"]

assert fingerprint(original_body) == (
    FREEZE_FINGERPRINT
)

assert code_fingerprint() == (
    original_body["code_fingerprint"]
)

# ============================================================
# 4. Create grouped authorization directory
# ============================================================

GROUPED_AUTH_DIR = (
    FREEZE_DIR
    / "grouped_authorization"
    / datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )
)

GROUPED_AUTH_DIR.mkdir(
    parents=True,
    exist_ok=False
)

GROUPED_FREEZE_PATH = (
    GROUPED_AUTH_DIR
    / "frozen_grouped_config.json"
)

GROUPED_APPROVAL_PATH = (
    GROUPED_AUTH_DIR
    / "approval_grouped.json"
)

# ============================================================
# 5. Freeze grouped evaluation protocol
# ============================================================

GROUPED_BODY = dict(original_body)

GROUPED_BODY.update({
    "authorized_protocols": [
        "grouped_outer"
    ],

    "nested_selection_protocol": (
        "groupkfold_inner_full_matrix_v1"
    ),

    "chronological_selection_frozen": True,

    "original_chronological_freeze_fingerprint": (
        FREEZE_FINGERPRINT
    ),

    "grouped_authorized_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "grouped_protocol_details": {
        "outer_folds": 4,
        "inner_folds": CONFIG.inner_folds,
        "model_names": CONFIG.model_names,
        "feature_experiments": CONFIG.experiment_ids,
        "top_k": CONFIG.top_k,
        "trials_per_candidate": CONFIG.trials,
        "selection_metric": "inner_validation_mae",
        "heldout_selection_prohibited": True,
        "chronological_model_reselection": False
    }
})

assert GROUPED_BODY["selection"] == (
    original_body["selection"]
)

GROUPED_FREEZE_FINGERPRINT = fingerprint(
    GROUPED_BODY
)

GROUPED_FREEZE_PATH.write_text(
    json.dumps(
        {
            "frozen_payload": GROUPED_BODY,

            "freeze_fingerprint": (
                GROUPED_FREEZE_FINGERPRINT
            ),

            "note": (
                "Authorizes the original nested "
                "participant-independent protocol. "
                "Chronological model remains frozen."
            )
        },
        indent=2
    )
)

# ============================================================
# 6. Create scoped approval
# ============================================================

GROUPED_APPROVAL = json.loads(
    FINAL_APPROVAL_PATH.read_text()
)

GROUPED_APPROVAL.update({
    "approved": True,

    "approved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "authorized_scopes": [
        "final_execution",
        "grouped_outer"
    ],

    "approved_freeze_fingerprint": (
        GROUPED_FREEZE_FINGERPRINT
    ),

    "note": (
        "Authorizes nested participant-independent "
        "evaluation only. No chronological test rerun, "
        "sensitivity test, midpoint release, or export."
    )
})

GROUPED_APPROVAL_PATH.write_text(
    json.dumps(
        GROUPED_APPROVAL,
        indent=2
    )
)

# ============================================================
# 7. Construct grouped-only configuration
# ============================================================

GROUPED_CONFIG = replace(
    FINAL_CONFIG,

    mode="final",

    approval_record=str(
        GROUPED_APPROVAL_PATH
    ),

    frozen_config=str(
        GROUPED_FREEZE_PATH
    ),

    resume_dir=None,

    run_final_test=False,

    run_grouped_outer=True,

    run_sensitivity_test=False,

    run_export=False
)

# Preserve original scientific configuration.

assert GROUPED_CONFIG.config_fingerprint == (
    CONFIG.config_fingerprint
)

assert GROUPED_CONFIG.inner_folds == 3
assert GROUPED_CONFIG.top_k == 3
assert GROUPED_CONFIG.trials == 30

assert GROUPED_CONFIG.run_tuning

assert not GROUPED_CONFIG.fast_mode

# ============================================================
# 8. Verify authorization gates
# ============================================================

require_approval(
    GROUPED_CONFIG,
    BUNDLE,
    "final_execution"
)

verified_grouped = require_freeze(
    GROUPED_CONFIG,
    BUNDLE,
    "grouped_outer"
)

assert fingerprint(verified_grouped) == (
    GROUPED_FREEZE_FINGERPRINT
)

# Save execution configuration.

(
    GROUPED_AUTH_DIR
    / "grouped_execution_config.json"
).write_text(
    json.dumps(
        GROUPED_CONFIG.payload(),
        indent=2
    )
)

print("=" * 75)
print("GROUPED EVALUATION AUTHORIZED")
print("=" * 75)

print("Outer folds: 4")
print("Inner folds:", GROUPED_CONFIG.inner_folds)
print("Model families:", GROUPED_CONFIG.model_names)

print(
    "Feature experiments:",
    GROUPED_CONFIG.experiment_ids
)

print("Top candidates:", GROUPED_CONFIG.top_k)
print("Trials per candidate:", GROUPED_CONFIG.trials)

print(
    "\nGrouped freeze fingerprint:",
    GROUPED_FREEZE_FINGERPRINT
)

print(
    "Configuration fingerprint:",
    GROUPED_CONFIG.config_fingerprint
)

print("\nGrouped approval:", GROUPED_APPROVAL_PATH)

print("\nGrouped evaluation authorized: YES")
print("Chronological test rerun authorized: NO")
print("Original model changed: NO")
print("Model export authorized: NO")

print("\nGrouped evaluation actually executed: NO")

print("=" * 75)


In [ ]:

# ============================================================
# RUN PARTICIPANT-INDEPENDENT GROUPED EVALUATION
# ============================================================

import json
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from IPython.display import display

from ml.sleep.train import run
from ml.sleep.config import (
    require_approval,
    require_freeze,
    file_hash
)

# ============================================================
# 1. Check authorization
# ============================================================

assert GROUPED_CONFIG.mode == "final"

assert GROUPED_CONFIG.run_grouped_outer

assert not GROUPED_CONFIG.run_final_test
assert not GROUPED_CONFIG.run_sensitivity_test
assert not GROUPED_CONFIG.run_export

require_approval(
    GROUPED_CONFIG,
    BUNDLE,
    "final_execution"
)

require_freeze(
    GROUPED_CONFIG,
    BUNDLE,
    "grouped_outer"
)

assert file_hash(CHECKPOINT_PATH) == (
    CHECKPOINT_SHA256
)

# ============================================================
# 2. Prevent accidental repeated execution
# ============================================================

GROUPED_STARTED_MARKER = (
    GROUPED_AUTH_DIR
    / "grouped_started.json"
)

GROUPED_COMPLETED_MARKER = (
    GROUPED_AUTH_DIR
    / "grouped_completed.json"
)

assert not GROUPED_STARTED_MARKER.exists(), (
    "Grouped evaluation already initiated. "
    "Inspect the previous execution before retrying."
)

started_at = datetime.now(
    timezone.utc
).isoformat()

GROUPED_STARTED_MARKER.write_text(
    json.dumps(
        {
            "status": "started",
            "started_at_utc": started_at,
            "freeze_fingerprint": (
                GROUPED_FREEZE_FINGERPRINT
            )
        },
        indent=2
    )
)

print("=" * 75)
print("NESTED GROUPED EVALUATION")
print("=" * 75)

print("Outer folds: 4")
print("Inner folds: 3")
print("Initial configurations per outer fold: 30")
print("Tuned candidates per outer fold: 3")
print("Trials per tuned candidate: 30")

print("\nExpected maximum model fits: 1444")

print("Started:", started_at)

print("\nOriginal chronological checkpoint unchanged.")

print("=" * 75, flush=True)

# ============================================================
# 3. Run official Agent 4 grouped evaluator
# ============================================================

start_time = perf_counter()

GROUPED_RUN_DIR = Path(
    run(GROUPED_CONFIG)
)

elapsed = perf_counter() - start_time

# ============================================================
# 4. Verify completed outputs
# ============================================================

fold_metrics_path = (
    GROUPED_RUN_DIR
    / "metrics"
    / "grouped_fold_results.csv"
)

pooled_metrics_path = (
    GROUPED_RUN_DIR
    / "metrics"
    / "grouped_pooled_metrics.json"
)

assert fold_metrics_path.is_file()
assert pooled_metrics_path.is_file()

grouped_folds = pd.read_csv(
    fold_metrics_path
)

grouped_pooled = json.loads(
    pooled_metrics_path.read_text()
)

assert len(grouped_folds) == 4

assert grouped_folds[
    "outer_test_fold"
].nunique() == 4

assert file_hash(CHECKPOINT_PATH) == (
    CHECKPOINT_SHA256
)

# ============================================================
# 5. Record successful completion
# ============================================================

GROUPED_COMPLETED_MARKER.write_text(
    json.dumps(
        {
            "status": "completed",
            "started_at_utc": started_at,

            "finished_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),

            "grouped_run_dir": str(
                GROUPED_RUN_DIR
            ),

            "freeze_fingerprint": (
                GROUPED_FREEZE_FINGERPRINT
            ),

            "runtime_seconds": elapsed,

            "completed_outer_folds": 4,

            "original_checkpoint_sha256": (
                CHECKPOINT_SHA256
            )
        },
        indent=2
    )
)

# ============================================================
# 6. Display final grouped metrics
# ============================================================

metrics = grouped_pooled["metrics"]

print("\n" + "=" * 75)
print("GROUPED EVALUATION COMPLETED")
print("=" * 75)

print("Completed outer folds:", len(grouped_folds))

print("\nPooled held-out metrics:")

print(
    json.dumps(
        metrics,
        indent=2
    )
)

print("\nOUTER FOLD RESULTS")

display(grouped_folds)

print(
    "\nTotal runtime:",
    round(elapsed / 60, 2),
    "minutes"
)

print("\nGrouped run directory:", GROUPED_RUN_DIR)

print("\nOriginal model changed: NO")
print("Chronological test rerun: NO")

print("=" * 75)


## 12. Separate scenario sensitivities

Reference offset0/lag1, UTC+1, UTC+2 and wellness t−2 retain separate provided manifests/fingerprints. Refit frozen family/config on allowed scenario train data, compare full/matched validation keys and missingness. Protected sensitivity test is separately off/gated. Offsets are not recovered timezones; lag2 does not prove ingestion.

In [ ]:

# ============================================================
# AGENT 4 — SENSITIVITY SCENARIO PREFLIGHT
# No model training or protected evaluation
# ============================================================

import json
import pandas as pd

from pathlib import Path
from dataclasses import replace
from IPython.display import display

from ml.sleep.data_validation import load_bundle
from ml.sleep.features import select_features
from ml.sleep.config import (
    require_freeze,
    file_hash
)

# ============================================================
# 1. Verify original frozen checkpoint
# ============================================================

assert GROUPED_COMPLETED_MARKER.is_file()
assert TEST_COMPLETED_MARKER.is_file()

assert file_hash(CHECKPOINT_PATH) == (
    CHECKPOINT_SHA256
)

require_freeze(
    FINAL_CONFIG,
    BUNDLE,
    "final_test"
)

# ============================================================
# 2. Locate sensitivity scenarios
# ============================================================

SCENARIO_NAMES = [
    "offset_plus1",
    "offset_plus2",
    "wellness_lag2"
]

SCENARIO_ROOT = (
    Path(CONFIG.data_dir)
    / "sensitivity"
)

REQUIRED_SCENARIO_FILES = [
    "sleep_dataset.parquet",
    "shared_features.parquet",
    "split_manifest.csv",
    "grouped_split_manifest.csv",
    "preprocessing_run.json"
]

SCENARIO_BUNDLES = {}
SCENARIO_FINGERPRINTS = {}
SCENARIO_SUMMARY = []

print("=" * 75)
print("SENSITIVITY SCENARIO PREFLIGHT")
print("=" * 75)

print("Scenario root:", SCENARIO_ROOT)

# ============================================================
# 3. Validate each scenario independently
# ============================================================

for name in SCENARIO_NAMES:

    scenario_path = SCENARIO_ROOT / name

    print("\n" + "-" * 70)
    print("Scenario:", name)
    print("Directory:", scenario_path)
    print("-" * 70)

    missing_files = [
        filename
        for filename in REQUIRED_SCENARIO_FILES
        if not (scenario_path / filename).is_file()
    ]

    if missing_files:

        print("STATUS: MISSING INPUTS")

        for filename in missing_files:
            print("[MISSING]", filename)

        SCENARIO_SUMMARY.append({
            "scenario": name,
            "status": "missing_inputs",
            "missing_files": ", ".join(missing_files)
        })

        continue

    try:

        scenario_config = replace(
            CONFIG,
            scenario_id=name,
            data_dir=str(scenario_path)
        )

        scenario_bundle = load_bundle(
            scenario_config
        )

        assert scenario_bundle.report["status"] == (
            "software_contract_valid"
        )

        assert all(
            scenario_bundle.report["checks"].values()
        )

        assert (
            scenario_bundle.report["protected_outcomes_disclosed"]
            is False
        )

        # Verify the frozen feature order remains valid.

        columns = select_features(
            scenario_bundle.schema,
            SELECTION["experiment_id"]
        )

        assert columns == (
            SELECTION["ordered_features"]
        )

        SCENARIO_BUNDLES[name] = (
            scenario_bundle
        )

        SCENARIO_FINGERPRINTS[name] = (
            scenario_bundle.fingerprints
        )

        split_counts = scenario_bundle.report[
            "split_row_counts"
        ]

        SCENARIO_SUMMARY.append({
            "scenario": name,
            "status": "software_contract_valid",
            "rows": scenario_bundle.report["rows"],
            "participants": scenario_bundle.report[
                "participants"
            ],
            "train_rows": split_counts.get("train", 0),
            "validation_rows": split_counts.get(
                "validation", 0
            ),
            "test_rows": split_counts.get("test", 0),
            "n_features": len(columns),
            "all_checks_passed": True
        })

        print("STATUS: VALID")

        print("Total rows:", scenario_bundle.report["rows"])
        print("Split counts:", split_counts)

        print(
            "Feature order:",
            "VERIFIED"
        )

        print("\nFingerprints:")

        print(
            json.dumps(
                scenario_bundle.fingerprints,
                indent=2
            )
        )

    except Exception as exc:

        SCENARIO_SUMMARY.append({
            "scenario": name,
            "status": "validation_failed",
            "error": repr(exc)
        })

        print("VALIDATION FAILED:", repr(exc))

# ============================================================
# 4. Save aggregate preflight report
# ============================================================

SENSITIVITY_PREFLIGHT_DIR = (
    FREEZE_DIR
    / "sensitivity_preflight"
)

SENSITIVITY_PREFLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

scenario_summary_df = pd.DataFrame(
    SCENARIO_SUMMARY
)

scenario_summary_df.to_csv(
    SENSITIVITY_PREFLIGHT_DIR
    / "scenario_summary.csv",
    index=False
)

(
    SENSITIVITY_PREFLIGHT_DIR
    / "scenario_fingerprints.json"
).write_text(
    json.dumps(
        SCENARIO_FINGERPRINTS,
        indent=2
    )
)

print("\n" + "=" * 75)
print("SENSITIVITY PREFLIGHT SUMMARY")
print("=" * 75)

display(scenario_summary_df)

print(
    "\nValid scenarios:",
    len(SCENARIO_BUNDLES),
    "/",
    len(SCENARIO_NAMES)
)

print("Original checkpoint unchanged: YES")
print("Model training performed: NO")
print("Protected evaluation performed: NO")

if len(SCENARIO_BUNDLES) == 3:

    print("\nALL SENSITIVITY SCENARIOS READY")

else:

    print(
        "\nResolve missing or invalid scenarios "
        "before sensitivity evaluation."
    )



In [ ]:

# ============================================================
# AGENT 4 — SENSITIVITY VALIDATION
# Frozen CatBoost E5 configuration
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace
from time import perf_counter
from IPython.display import display

from ml.sleep.config import (
    TARGET,
    fingerprint,
    file_hash,
    require_approval,
    require_freeze
)

from ml.sleep.experiments import (
    sensitivity,
    load_frozen_model
)

from ml.sleep.train import (
    initialize_run,
    code_fingerprint
)

from ml.sleep.features import matrix
from ml.sleep.evaluation import regression_metrics
from ml.sleep.logging_utils import safe_json


# ============================================================
# 1. Verify completed stages and scenario preflight
# ============================================================

assert TEST_COMPLETED_MARKER.is_file()
assert GROUPED_COMPLETED_MARKER.is_file()

assert len(SCENARIO_BUNDLES) == 3
assert len(SCENARIO_FINGERPRINTS) == 3

assert all(
    bundle.report["status"] == "software_contract_valid"
    for bundle in SCENARIO_BUNDLES.values()
)

assert all(
    all(bundle.report["checks"].values())
    for bundle in SCENARIO_BUNDLES.values()
)

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256

original_body = require_freeze(
    FINAL_CONFIG,
    BUNDLE,
    "final_test"
)

assert fingerprint(original_body) == FREEZE_FINGERPRINT

assert original_body["code_fingerprint"] == (
    code_fingerprint()
)

assert original_body["selection"] == SELECTION


# ============================================================
# 2. Explicit sensitivity authorization
# ============================================================

ML_LEAD_APPROVES_SENSITIVITY = False
if not ML_LEAD_APPROVES_SENSITIVITY:

    raise PermissionError(
        "Review all three scenario validation reports "
        "and authorize sensitivity validation first."
    )

SENSITIVITY_AUTH_DIR = (
    FREEZE_DIR
    / "sensitivity_authorization"
    / datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )
)

SENSITIVITY_AUTH_DIR.mkdir(
    parents=True,
    exist_ok=False
)

SENSITIVITY_APPROVAL_PATH = (
    SENSITIVITY_AUTH_DIR
    / "approval_sensitivity_validation.json"
)

SENSITIVITY_APPROVAL = json.loads(
    FINAL_APPROVAL_PATH.read_text()
)

SENSITIVITY_APPROVAL.update({

    "approved": True,

    "approved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "authorized_scopes": [
        "sensitivity_validation"
    ],

    "scenario_fingerprints": SCENARIO_FINGERPRINTS,

    "approved_freeze_fingerprint": (
        FREEZE_FINGERPRINT
    ),

    "note": (
        "Authorizes validation-only sensitivity "
        "analysis for offset_plus1, offset_plus2, "
        "and wellness_lag2 using the unchanged "
        "frozen CatBoost E5 configuration. "
        "No chronological test rerun, model "
        "reselection, protected sensitivity test, "
        "or production export."
    )

})

SENSITIVITY_APPROVAL_PATH.write_text(
    json.dumps(
        SENSITIVITY_APPROVAL,
        indent=2
    )
)


# ============================================================
# 3. Construct sensitivity execution configuration
# ============================================================

# Preserve the original scientific configuration.
# Call the official sensitivity function directly instead
# of invoking the complete development runner.

SENSITIVITY_CONFIG = replace(
    CONFIG,

    mode="develop",

    approval_record=str(
        SENSITIVITY_APPROVAL_PATH
    ),

    frozen_config=str(
        FREEZE_PATH
    ),

    resume_dir=None,

    run_final_test=False,

    run_grouped_outer=False,

    run_sensitivity_test=False,

    run_export=False
)

assert SENSITIVITY_CONFIG.config_fingerprint == (
    CONFIG.config_fingerprint
)

require_approval(
    SENSITIVITY_CONFIG,
    BUNDLE,
    "sensitivity_validation"
)

assert SENSITIVITY_APPROVAL[
    "scenario_fingerprints"
] == SCENARIO_FINGERPRINTS

print("=" * 75)
print("SENSITIVITY VALIDATION AUTHORIZED")
print("=" * 75)

print(
    "Scenarios:",
    list(SCENARIO_FINGERPRINTS)
)

print("Model:", SELECTION["model_name"])
print("Features:", SELECTION["experiment_id"])

print(
    "Fixed parameters:",
    SELECTION["params"]
)

print("Protected sensitivity test: NO")
print("Model reselection: NO")
print("Original model retraining: NO")


# ============================================================
# 4. Initialize official experiment logger
# ============================================================

sensitivity_logger = initialize_run(
    SENSITIVITY_CONFIG,
    BUNDLE,
    label="sensitivity_validation_only"
)

SENSITIVITY_RUN_DIR = Path(
    sensitivity_logger.root
)

sensitivity_logger.write_json(
    "metadata/sensitivity_protocol.json",
    {
        "reference_freeze_fingerprint": FREEZE_FINGERPRINT,
        "reference_checkpoint_sha256": CHECKPOINT_SHA256,
        "scenario_fingerprints": SCENARIO_FINGERPRINTS,
        "selection": SELECTION,
        "evaluation_partition": "validation",
        "test_used": False,
        "model_reselection": False
    }
)

start_time = perf_counter()

print("\n" + "=" * 75)
print("STARTING SENSITIVITY EVALUATION")
print("=" * 75)

print(
    "Started at:",
    datetime.now(timezone.utc).isoformat()
)

print("Output directory:", SENSITIVITY_RUN_DIR)

print(
    "Planned scenario model fits:",
    len(SCENARIO_BUNDLES)
)

print("=" * 75, flush=True)


# ============================================================
# 5. Run official scenario sensitivity evaluation
# ============================================================

try:

    with sensitivity_logger.event(
        "sensitivity_validation_suite",
        "suite",
        requested_device=SENSITIVITY_CONFIG.device
    ):

        sensitivity_results = sensitivity(
            bundle=BUNDLE,
            config=SENSITIVITY_CONFIG,
            logger=sensitivity_logger,
            selection=SELECTION,
            protected=False
        )

finally:

    sensitivity_logger.summary()

assert len(sensitivity_results) == 3

assert {
    row["scenario_id"]
    for row in sensitivity_results
} == set(SCENARIO_BUNDLES)

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256


# ============================================================
# 6. Reference validation performance
# ============================================================

_, reference_model, reference_columns = (
    load_frozen_model(
        FINAL_CONFIG,
        BUNDLE,
        "final_test"
    )
)

reference_val = BUNDLE.partition(
    "validation"
).reset_index(drop=True)

assert len(reference_val) == 329

reference_predictions = reference_model.predict(
    matrix(
        reference_val,
        reference_columns
    )
)

reference_metrics, _ = regression_metrics(
    reference_val[TARGET],
    reference_predictions,
    reference_val["participant_id"],
    reference_val["sleep_tst_prev"]
)

assert np.isclose(
    reference_metrics["mae_minutes"],
    SELECTION["mae_minutes"],
    atol=1e-5
)

# ============================================================
# 7. Matched-key reference comparison
# ============================================================

# Compare each scenario with the reference model using
# exactly the same matched participant/date keys.

matched_results = []

sensitivity_table = pd.DataFrame(
    sensitivity_results
)

for scenario_name, scenario_bundle in (
    SCENARIO_BUNDLES.items()
):

    scenario_val = scenario_bundle.partition(
        "validation"
    ).reset_index(drop=True)

    keys = [
        "participant_id",
        "cutoff_date"
    ]

    reference_matched = reference_val.merge(
        scenario_val[keys],
        on=keys,
        validate="one_to_one"
    )

    assert len(reference_matched) > 0

    matched_reference_predictions = (
        reference_model.predict(
            matrix(
                reference_matched,
                reference_columns
            )
        )
    )

    matched_reference_metrics, _ = (
        regression_metrics(
            reference_matched[TARGET],
            matched_reference_predictions,
            reference_matched["participant_id"],
            reference_matched["sleep_tst_prev"]
        )
    )

    scenario_result = sensitivity_table.loc[
        sensitivity_table["scenario_id"]
        == scenario_name
    ].iloc[0]

    assert len(reference_matched) == (
        int(scenario_result["matched_n"])
    )

    ref_mae = matched_reference_metrics[
        "mae_minutes"
    ]

    scenario_mae = float(
        scenario_result["matched_mae_minutes"]
    )

    matched_results.append({

        "scenario_id": scenario_name,

        "matched_n": len(reference_matched),

        "reference_matched_mae_minutes": ref_mae,

        "scenario_matched_mae_minutes": scenario_mae,

        "mae_change_minutes": (
            scenario_mae - ref_mae
        ),

        "mae_change_pct": (
            100 * (scenario_mae - ref_mae) / ref_mae
        )

    })

matched_comparison = pd.DataFrame(
    matched_results
)

matched_comparison.to_csv(
    SENSITIVITY_RUN_DIR
    / "metrics"
    / "matched_reference_comparison.csv",
    index=False
)

# ============================================================
# 8. Consolidated sensitivity leaderboard
# ============================================================

reference_row = {

    "scenario_id": "primary",

    "partition": "validation",

    "reference_n": len(reference_val),

    "scenario_n": len(reference_val),

    "matched_n": len(reference_val),

    "reference_only": 0,

    "scenario_only": 0,

    "target_changes": 0,

    "changed_feature_values": 0,

    "reference_feature_missing_fraction": float(
        reference_val[
            reference_columns
        ].isna().mean().mean()
    ),

    "scenario_feature_missing_fraction": float(
        reference_val[
            reference_columns
        ].isna().mean().mean()
    ),

    "matched_mae_minutes": reference_metrics[
        "mae_minutes"
    ],

    **reference_metrics
}

sensitivity_leaderboard = pd.concat(
    [
        pd.DataFrame([reference_row]),
        sensitivity_table
    ],
    ignore_index=True
)

reference_mae = reference_metrics["mae_minutes"]

sensitivity_leaderboard[
    "mae_change_vs_reference_minutes"
] = (
    sensitivity_leaderboard["mae_minutes"]
    - reference_mae
)

sensitivity_leaderboard[
    "mae_change_vs_reference_pct"
] = (
    sensitivity_leaderboard[
        "mae_change_vs_reference_minutes"
    ]
    / reference_mae
    * 100
)

sensitivity_leaderboard.to_csv(
    SENSITIVITY_RUN_DIR
    / "metrics"
    / "sensitivity_leaderboard.csv",
    index=False
)

# ============================================================
# 9. Display aggregate results
# ============================================================

print("\n" + "=" * 80)
print("SENSITIVITY VALIDATION LEADERBOARD")
print("=" * 80)

display_columns = [
    "scenario_id",
    "scenario_n",
    "mae_minutes",
    "rmse_minutes",
    "participant_macro_mae_minutes",
    "mae_change_vs_reference_minutes",
    "mae_change_vs_reference_pct"
]

display(
    sensitivity_leaderboard[
        display_columns
    ].round(4)
)

print("\nMATCHED-KEY COMPARISON")

display(
    matched_comparison.round(4)
)

print("\nDATASET AND FEATURE CHANGES")

display(
    sensitivity_leaderboard[
        [
            "scenario_id",
            "matched_n",
            "reference_only",
            "scenario_only",
            "target_changes",
            "changed_feature_values",
            "scenario_feature_missing_fraction"
        ]
    ].round(4)
)


# ============================================================
# 10. Feature-level sensitivity summary
# ============================================================

feature_change_records = []

for row in sensitivity_results:

    for feature, count in row[
        "feature_changes"
    ].items():

        feature_change_records.append({
            "scenario_id": row["scenario_id"],
            "feature_name": feature,
            "changed_matched_rows": count
        })

feature_changes = pd.DataFrame(
    feature_change_records
)

feature_changes.to_csv(
    SENSITIVITY_RUN_DIR
    / "metrics"
    / "feature_change_summary.csv",
    index=False
)

print("\nMOST AFFECTED FEATURES")

display(
    feature_changes.loc[
        feature_changes["changed_matched_rows"] > 0
    ].sort_values(
        "changed_matched_rows",
        ascending=False
    ).head(25)
)


# ============================================================
# 11. Save completion summary
# ============================================================

elapsed = perf_counter() - start_time

sensitivity_summary = {

    "status": "completed",

    "completed_scenarios": len(
        sensitivity_results
    ),

    "reference_mae_minutes": reference_mae,

    "reference_checkpoint_sha256": CHECKPOINT_SHA256,

    "reference_freeze_fingerprint": FREEZE_FINGERPRINT,

    "scenario_fingerprints": SCENARIO_FINGERPRINTS,

    "scenario_results": sensitivity_results,

    "matched_reference_comparison": (
        matched_comparison.to_dict(
            orient="records"
        )
    ),

    "runtime_seconds": elapsed,

    "original_checkpoint_unchanged": True,

    "model_reselection": False,

    "protected_test_used": False,

    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat()

}

SENSITIVITY_COMPLETED_MARKER = (
    SENSITIVITY_AUTH_DIR
    / "sensitivity_completed.json"
)

SENSITIVITY_COMPLETED_MARKER.write_text(
    json.dumps(
        safe_json(sensitivity_summary),
        indent=2,
        allow_nan=False
    )
)

print("\n" + "=" * 75)
print("SENSITIVITY EVALUATION COMPLETED")
print("=" * 75)

print(
    "Completed scenarios:",
    len(sensitivity_results)
)

print(
    "Total runtime:",
    round(elapsed, 2),
    "seconds"
)

print("\nOriginal checkpoint unchanged: YES")
print("Model reselection: NO")
print("Protected test evaluated: NO")

print(
    "\nResults directory:",
    SENSITIVITY_RUN_DIR
)

print("=" * 75)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Image

sensitivity_leaderboard = pd.read_csv(
    SENSITIVITY_RUN_DIR
    / "metrics"
    / "sensitivity_leaderboard.csv"
)

labels = {
    "primary": "Reference (UTC+0)",
    "offset_plus1": "Wellness UTC+1",
    "offset_plus2": "Wellness UTC+2",
    "wellness_lag2": "Wellness t-2"
}

plot_data = sensitivity_leaderboard.copy()

plot_data["label"] = (
    plot_data["scenario_id"].map(labels)
)

reference_mae = plot_data.loc[
    plot_data["scenario_id"] == "primary",
    "mae_minutes"
].iloc[0]

# ============================================================
# 1. MAE comparison
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))

colors = [
    "#7570B3" if name == "primary"
    else "#2A9D8F"
    for name in plot_data["scenario_id"]
]

bars = ax.barh(
    plot_data["label"],
    plot_data["mae_minutes"],
    color=colors
)

ax.bar_label(
    bars,
    fmt="%.3f",
    padding=5
)

ax.axvline(
    reference_mae,
    linestyle="--",
    color="#7570B3",
    label="Reference MAE"
)

ax.set_xlabel("Validation MAE (minutes)")

ax.set_title(
    "Sleep Prediction — Preprocessing Sensitivity"
)

ax.set_xlim(
    0,
    plot_data["mae_minutes"].max() * 1.18
)

ax.invert_yaxis()
ax.legend()
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

mae_path = (
    SENSITIVITY_RUN_DIR
    / "figures"
    / "sensitivity_mae_comparison.png"
)

fig.savefig(
    mae_path,
    dpi=160,
    bbox_inches="tight"
)

plt.close(fig)

display(
    Image(filename=str(mae_path))
)


# ============================================================
# 2. MAE change relative to reference
# ============================================================

comparison = plot_data.loc[
    plot_data["scenario_id"] != "primary"
].copy()

fig, ax = plt.subplots(figsize=(10, 5))

changes = comparison[
    "mae_change_vs_reference_minutes"
]

bars = ax.barh(
    comparison["label"],
    changes,
    color=[
        "#D95F02" if value > 0
        else "#1B9E77"
        for value in changes
    ]
)

ax.bar_label(
    bars,
    fmt="%+.3f",
    padding=5
)

ax.axvline(
    0,
    color="black",
    linewidth=1
)

ax.set_xlabel(
    "MAE change versus reference (minutes)"
)

ax.set_title(
    "Sensitivity Impact on Sleep Prediction Error"
)

ax.margins(x=0.25)
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

delta_path = (
    SENSITIVITY_RUN_DIR
    / "figures"
    / "sensitivity_mae_change.png"
)

fig.savefig(
    delta_path,
    dpi=160,
    bbox_inches="tight"
)

plt.close(fig)

display(
    Image(filename=str(delta_path))
)


# ============================================================
# 3. Feature changes and missingness
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    plot_data["label"],
    plot_data[
        "scenario_feature_missing_fraction"
    ] * 100
)

ax.bar_label(
    bars,
    fmt="%.2f%%",
    padding=5
)

ax.set_xlabel("Missing feature values (%)")

ax.set_title(
    "Feature Missingness Across Sensitivity Scenarios"
)

ax.set_xlim(
    0,
    max(
        1,
        plot_data[
            "scenario_feature_missing_fraction"
        ].max() * 125
    )
)

ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

missingness_path = (
    SENSITIVITY_RUN_DIR
    / "figures"
    / "sensitivity_missingness.png"
)

fig.savefig(
    missingness_path,
    dpi=160,
    bbox_inches="tight"
)

plt.close(fig)

display(
    Image(filename=str(missingness_path))
)

print("\nAll sensitivity figures saved.")


## 13. Optional midpoint

Disabled until human-recorded primary TST completion. Separate sin/cos regressors represent the full interval midpoint, decode atan2 modulo 1440, abstain for near-zero/nonfinite vectors, and compare circular error with previous/reference timing on matched available rows. No biological phase, bedtime or wake time is inferred. Current optional validation experiment is predeclared Ridge E0; protected timing release needs a separate reviewed freeze.

In [ ]:
assert not CONFIG.run_midpoint

print("Optional midpoint prediction: SKIPPED")
print("Reason: Primary TST model is the MVP priority")
print("Midpoint validated: False")
print("Midpoint output: None")


## 14. Explainability, uncertainty, export and inference

Validation permutation importance and scaled Ridge coefficients are predictive associations, not causes. Final errors/cluster bootstrap appear only after authorization. Export is separately off and verifies CPU reload consistency. Agents 5/6 receive an independent CPU helper with null/unavailable/insufficient_data semantics, exact schema/order and metadata. No confidence is inferred from risk, coverage or vector norm.

In [ ]:

# ============================================================
# AGENT 4 — FINAL MODEL SHAP EXPLAINABILITY
# Frozen CatBoost E5 | Validation partition only
# ============================================================

import json
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from IPython.display import display
from catboost import Pool

from ml.sleep.config import (
    file_hash,
    require_freeze
)

from ml.sleep.features import matrix

# ============================================================
# 1. Verify frozen model
# ============================================================

assert TEST_COMPLETED_MARKER.is_file()
assert GROUPED_COMPLETED_MARKER.is_file()
assert SENSITIVITY_COMPLETED_MARKER.is_file()

require_freeze(
    FINAL_CONFIG,
    BUNDLE,
    "final_test"
)

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256

SHAP_MODEL = joblib.load(
    CHECKPOINT_PATH
).cpu_copy()

assert SHAP_MODEL.model_name == "catboost"

SHAP_FEATURES = list(
    SELECTION["ordered_features"]
)

assert SHAP_FEATURES == list(
    SHAP_MODEL.columns
)

assert len(SHAP_FEATURES) == 68

# ============================================================
# 2. Prepare explanation directory
# ============================================================

SHAP_DIR = (
    Path(DEV_RUN_DIR)
    / "final_explainability"
    / datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )
)

SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STARTED_AT = datetime.now(
    timezone.utc
).isoformat()

start_time = perf_counter()

print("=" * 75)
print("FROZEN CATBOOST SHAP EXPLAINABILITY")
print("=" * 75)

print("Model: CatBoost E5 Tuned")
print("Features:", len(SHAP_FEATURES))
print("Device: CPU")
print("Partition: validation")
print("Started:", STARTED_AT)

# ============================================================
# 3. Load validation features only
# ============================================================

shap_frame = BUNDLE.partition(
    "validation"
).reset_index(drop=True)

assert len(shap_frame) == 329

X_shap = matrix(
    shap_frame,
    SHAP_FEATURES
)

# Preserve exactly the preprocessing used by
# FittedRegressor.predict().

X_native = X_shap.to_numpy(
    dtype=float
)

if SHAP_MODEL.preprocessor is not None:

    X_native = SHAP_MODEL.preprocessor.transform(
        X_native
    )

assert X_native.shape == (
    329,
    68
)

shap_pool = Pool(
    X_native
)

print("Explanation samples:", len(X_shap))

# ============================================================
# 4. Compute native CatBoost TreeSHAP
# ============================================================

shap_start = perf_counter()

native_shap = np.asarray(

    SHAP_MODEL.estimator.get_feature_importance(
        data=shap_pool,
        type="ShapValues",
        thread_count=CONFIG.cpu_threads,
        verbose=False
    ),

    dtype=float
)

shap_runtime = (
    perf_counter() - shap_start
)

# CatBoost returns:
# first 68 columns = feature contributions
# last column = expected model prediction

assert native_shap.shape == (
    len(X_shap),
    len(SHAP_FEATURES) + 1
)

assert np.isfinite(native_shap).all()

shap_values = native_shap[:, :-1]

base_values = native_shap[:, -1]

print("\nSHAP computation completed")

print("SHAP matrix shape:", shap_values.shape)

print(
    "SHAP runtime:",
    round(shap_runtime, 3),
    "seconds"
)

# ============================================================
# 5. Verify SHAP additivity
# ============================================================

model_predictions = SHAP_MODEL.predict(
    X_shap
)

reconstructed_predictions = (
    base_values
    + shap_values.sum(axis=1)
)

np.testing.assert_allclose(
    reconstructed_predictions,
    model_predictions,
    rtol=1e-6,
    atol=1e-4
)

max_additivity_error = float(
    np.max(
        np.abs(
            reconstructed_predictions
            - model_predictions
        )
    )
)

print("\nSHAP additivity: VERIFIED")

print(
    "Maximum reconstruction error:",
    max_additivity_error
)

print(
    "Mean expected prediction:",
    round(float(base_values.mean()), 4),
    "minutes"
)

# ============================================================
# 6. Global feature importance
# ============================================================

feature_importance = pd.DataFrame({

    "feature": SHAP_FEATURES,

    "mean_abs_shap_minutes": np.mean(
        np.abs(shap_values),
        axis=0
    ),

    "mean_signed_shap_minutes": np.mean(
        shap_values,
        axis=0
    ),

    "std_shap_minutes": np.std(
        shap_values,
        axis=0
    )

})

schema_info = BUNDLE.schema[
    ["name", "group", "unit"]
].rename(
    columns={"name": "feature"}
)

feature_importance = feature_importance.merge(
    schema_info,
    on="feature",
    how="left",
    validate="one_to_one"
)

assert feature_importance["group"].notna().all()

total_importance = feature_importance[
    "mean_abs_shap_minutes"
].sum()

feature_importance["relative_importance_pct"] = (

    feature_importance["mean_abs_shap_minutes"]

    / total_importance

    * 100
)

feature_importance = feature_importance.sort_values(
    "mean_abs_shap_minutes",
    ascending=False
).reset_index(drop=True)

feature_importance.insert(
    0,
    "rank",
    range(1, len(feature_importance) + 1)
)

feature_importance.to_csv(
    SHAP_DIR / "shap_global_importance.csv",
    index=False
)

print("\n" + "=" * 75)
print("TOP 20 GLOBAL SHAP FEATURES")
print("=" * 75)

display(
    feature_importance.head(20).round(4)
)

# ============================================================
# 7. Feature group importance
# ============================================================

group_names = {

    "A": "Sleep History",

    "B": "Wellness",

    "C": "Physiology",

    "D": "Activity",

    "E": "Personalization"
}

schema_group_map = (
    BUNDLE.schema
    .set_index("name")["group"]
    .to_dict()
)

group_records = []

for group_id in ["A", "B", "C", "D", "E"]:

    indices = [

        index

        for index, feature in enumerate(
            SHAP_FEATURES
        )

        if schema_group_map[feature] == group_id

    ]

    assert len(indices) > 0

    group_shap = shap_values[:, indices]

    # Sum of individual feature importances.

    sum_abs_importance = float(

        np.mean(
            np.abs(group_shap),
            axis=0
        ).sum()

    )

    # Net contribution after within-group cancellation.

    net_contribution = group_shap.sum(
        axis=1
    )

    group_records.append({

        "group": group_id,

        "group_name": group_names[group_id],

        "n_features": len(indices),

        "sum_mean_abs_shap_minutes": (
            sum_abs_importance
        ),

        "mean_abs_net_group_shap_minutes": float(

            np.mean(
                np.abs(net_contribution)
            )

        ),

        "mean_signed_group_shap_minutes": float(

            np.mean(net_contribution)

        )

    })

group_importance = pd.DataFrame(
    group_records
)

group_importance["importance_share_pct"] = (

    group_importance[
        "sum_mean_abs_shap_minutes"
    ]

    / group_importance[
        "sum_mean_abs_shap_minutes"
    ].sum()

    * 100

)

group_importance = group_importance.sort_values(

    "sum_mean_abs_shap_minutes",

    ascending=False

).reset_index(drop=True)

group_importance.to_csv(

    SHAP_DIR / "shap_group_importance.csv",

    index=False

)

print("\n" + "=" * 75)
print("FEATURE GROUP SHAP IMPORTANCE")
print("=" * 75)

display(
    group_importance.round(4)
)

# ============================================================
# 8. Save explanation metadata
# ============================================================

elapsed = perf_counter() - start_time

shap_summary = {

    "status": "completed",

    "method": "CatBoost native TreeSHAP",

    "model": "catboost",

    "feature_set": "E5",

    "partition": "validation",

    "n_samples": len(X_shap),

    "n_features": len(SHAP_FEATURES),

    "n_participants": int(
        shap_frame.participant_id.nunique()
    ),

    "prediction_unit": "minutes",

    "mean_expected_prediction_minutes": float(
        base_values.mean()
    ),

    "additivity_verified": True,

    "max_additivity_error": (
        max_additivity_error
    ),

    "top_features": (
        feature_importance.head(10)
        .to_dict(orient="records")
    ),

    "group_importance": (
        group_importance
        .to_dict(orient="records")
    ),

    "freeze_fingerprint": FREEZE_FINGERPRINT,

    "checkpoint_sha256": CHECKPOINT_SHA256,

    "shap_runtime_seconds": shap_runtime,

    "total_runtime_seconds": elapsed,

    "started_at_utc": STARTED_AT,

    "finished_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "test_used": False,

    "model_retrained": False,

    "interpretation": (
        "SHAP values explain the contribution of "
        "input features to model predictions. "
        "They do not establish causal effects, "
        "clinical confidence or medical recommendations."
    )

}

(
    SHAP_DIR / "shap_summary.json"
).write_text(
    json.dumps(
        shap_summary,
        indent=2,
        allow_nan=False
    )
)

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256

print("\n" + "=" * 75)
print("SHAP EXPLAINABILITY COMPLETED")
print("=" * 75)

print(
    "Total runtime:",
    round(elapsed, 3),
    "seconds"
)

print("Model unchanged: YES")
print("Model retrained: NO")
print("Protected test reused: NO")

print("\nResults directory:", SHAP_DIR)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Image

# ============================================================
# 1. Global SHAP importance
# ============================================================

top_features = feature_importance.head(
    15
).sort_values(
    "mean_abs_shap_minutes",
    ascending=True
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

bars = ax.barh(

    top_features["feature"],

    top_features["mean_abs_shap_minutes"],

    color="#2A9D8F"

)

ax.bar_label(
    bars,
    fmt="%.3f",
    padding=4
)

ax.set_xlabel(
    "Mean absolute SHAP value (minutes)"
)

ax.set_title(
    "Frozen CatBoost E5 — Global SHAP Importance"
)

ax.set_xlim(

    0,

    top_features[
        "mean_abs_shap_minutes"
    ].max() * 1.2

)

ax.grid(
    axis="x",
    alpha=0.2
)

fig.tight_layout()

global_path = (
    SHAP_DIR / "shap_global_importance.png"
)

fig.savefig(

    global_path,

    dpi=170,

    bbox_inches="tight"

)

plt.close(fig)

display(
    Image(filename=str(global_path))
)

# ============================================================
# 2. Feature group importance
# ============================================================

group_plot = group_importance.sort_values(

    "importance_share_pct",

    ascending=True

)

fig, ax = plt.subplots(
    figsize=(10, 5)
)

bars = ax.barh(

    group_plot["group_name"],

    group_plot["importance_share_pct"],

    color="#5B8DEF"

)

ax.bar_label(

    bars,

    fmt="%.1f%%",

    padding=5

)

ax.set_xlabel(
    "Share of total mean absolute SHAP (%)"
)

ax.set_title(
    "Contribution of Feature Families to Model Predictions"
)

ax.set_xlim(

    0,

    group_plot[
        "importance_share_pct"
    ].max() * 1.2

)

ax.grid(
    axis="x",
    alpha=0.2
)

fig.tight_layout()

group_path = (
    SHAP_DIR / "shap_group_importance.png"
)

fig.savefig(

    group_path,

    dpi=170,

    bbox_inches="tight"

)

plt.close(fig)

display(
    Image(filename=str(group_path))
)

# ============================================================
# 3. Aggregate SHAP direction summary
# ============================================================

# Show contribution distributions without publishing
# individual participant rows.

direction_features = (
    feature_importance
    .head(10)["feature"]
    .tolist()
)

direction_records = []

for feature in direction_features:

    index = SHAP_FEATURES.index(
        feature
    )

    values = shap_values[:, index]

    direction_records.append({

        "feature": feature,

        "median_shap": float(
            np.median(values)
        ),

        "q10": float(
            np.quantile(values, 0.10)
        ),

        "q90": float(
            np.quantile(values, 0.90)
        ),

        "mean_shap": float(
            np.mean(values)
        )

    })

direction_summary = pd.DataFrame(
    direction_records
)

direction_summary.to_csv(

    SHAP_DIR / "shap_direction_summary.csv",

    index=False

)

plot_data = direction_summary.iloc[
    ::-1
].reset_index(drop=True)

fig, ax = plt.subplots(
    figsize=(11, 6)
)

y_positions = np.arange(
    len(plot_data)
)

lower_error = (

    plot_data["median_shap"]

    - plot_data["q10"]

)

upper_error = (

    plot_data["q90"]

    - plot_data["median_shap"]

)

ax.errorbar(

    plot_data["median_shap"],

    y_positions,

    xerr=[
        lower_error,
        upper_error
    ],

    fmt="o",

    capsize=4,

    color="#2A9D8F",

    label="Median and 10th–90th percentiles"

)

ax.axvline(

    0,

    color="black",

    linestyle="--",

    linewidth=1

)

ax.set_yticks(
    y_positions
)

ax.set_yticklabels(
    plot_data["feature"]
)

ax.set_xlabel(
    "SHAP contribution to predicted TST (minutes)"
)

ax.set_title(
    "Distribution of Feature Contributions"
)

ax.legend()

ax.grid(
    axis="x",
    alpha=0.2
)

fig.tight_layout()

direction_path = (
    SHAP_DIR / "shap_direction_summary.png"
)

fig.savefig(

    direction_path,

    dpi=170,

    bbox_inches="tight"

)

plt.close(fig)

display(
    Image(filename=str(direction_path))
)

print("\nAll SHAP visualizations saved.")

print("Results directory:", SHAP_DIR)


## 15. Runtime and handoff

All output directories are unique and private. Do not publish participant predictions, models or executed notebook outputs. Save the complete private run directory to preserve Optuna and checkpoints across Kaggle sessions. Actual GPU/Kaggle execution has not been verified by Codex. PMData is terrestrial, correlated and wearable-estimated; unknown ingestion/timezones and license discrepancy remain.

In [ ]:
path = RUN_DIR / 'metrics/runtime_summary.csv'
if path.is_file():
    runtime = pd.read_csv(path)
    display(runtime[[c for c in ['experiment_id','stage','status','elapsed_seconds','requested_device','actual_device'] if c in runtime]])
print((RUN_DIR / 'evaluation_report.md').read_text())


In [ ]:

# ============================================================
# AGENT 4 — FINAL DELIVERY: OFFICIAL MODEL EXPORT
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace
from time import perf_counter

from ml.sleep.config import (
    fingerprint,
    file_hash,
    require_approval,
    require_freeze
)

from ml.sleep.train import run, code_fingerprint

from ml.sleep.logging_utils import safe_json


# ============================================================
# 1. Verify completed research stages
# ============================================================

assert TEST_COMPLETED_MARKER.is_file()
assert GROUPED_COMPLETED_MARKER.is_file()
assert SENSITIVITY_COMPLETED_MARKER.is_file()

assert (
    SHAP_DIR / "shap_summary.json"
).is_file()

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256

original_body = require_freeze(
    FINAL_CONFIG,
    BUNDLE,
    "final_test"
)

assert fingerprint(original_body) == FREEZE_FINGERPRINT

assert original_body["selection"] == SELECTION

assert original_body["code_fingerprint"] == (
    code_fingerprint()
)

assert SELECTION["model_name"] == "catboost"
assert SELECTION["experiment_id"] == "E5"
assert len(SELECTION["ordered_features"]) == 68

assert not CONFIG.run_midpoint

# Load actual completed evaluation results.

FINAL_TEST_METRICS = json.loads(
    (
        Path(FINAL_RUN_DIR)
        / "metrics/final_test_metrics.json"
    ).read_text()
)

GROUPED_RESULTS = json.loads(
    (
        Path(GROUPED_RUN_DIR)
        / "metrics/grouped_pooled_metrics.json"
    ).read_text()
)

SHAP_RESULTS = json.loads(
    (
        Path(SHAP_DIR)
        / "shap_summary.json"
    ).read_text()
)

SENSITIVITY_RESULTS = pd.read_csv(
    Path(SENSITIVITY_RUN_DIR)
    / "metrics/sensitivity_leaderboard.csv"
)

assert FINAL_TEST_METRICS["n"] == 296

assert FINAL_TEST_METRICS["freeze_fingerprint"] == (
    FREEZE_FINGERPRINT
)

assert SHAP_RESULTS["additivity_verified"] is True

assert SHAP_RESULTS["freeze_fingerprint"] == (
    FREEZE_FINGERPRINT
)

assert SHAP_RESULTS["checkpoint_sha256"] == (
    CHECKPOINT_SHA256
)

assert len(SENSITIVITY_RESULTS) == 4

print("All research stages verified.")

print(
    "Final test MAE:",
    FINAL_TEST_METRICS["mae_minutes"]
)

print(
    "Grouped pooled MAE:",
    GROUPED_RESULTS["metrics"]["mae_minutes"]
)


# ============================================================
# 2. Explicit private export approval
# ============================================================

# Set True only after reviewing the final results
# and confirming permitted private team use.

ML_LEAD_APPROVES_EXPORT = False
if not ML_LEAD_APPROVES_EXPORT:

    raise PermissionError(
        "Review completed evaluation results and "
        "authorize private research model export."
    )

APPROVED_BY = original_body["approved_by"]

EXPORT_TIMESTAMP = datetime.now(
    timezone.utc
).isoformat()

DELIVERY_ID = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%S%fZ")

DELIVERY_ROOT = (
    WORK_ROOT
    / "agent4_delivery"
    / DELIVERY_ID
)

DELIVERY_ROOT.mkdir(
    parents=True,
    exist_ok=False
)

EXPORT_AUTH_DIR = (
    DELIVERY_ROOT
    / "export_authorization"
)

EXPORT_AUTH_DIR.mkdir()


# ============================================================
# 3. Create export-only frozen authorization
# ============================================================

# Preserve original model selection and scientific identity.
# Do not modify the chronological freeze record.

EXPORT_BODY = dict(
    original_body
)

EXPORT_BODY.update({

    "authorized_protocols": ["export"],

    "original_chronological_freeze_fingerprint": (
        FREEZE_FINGERPRINT
    ),

    "production_export_authorized": True,

    "export_authorized_at_utc": EXPORT_TIMESTAMP,

    "release_scope": (
        "private_noncommercial_research_demo"
    )

})

assert EXPORT_BODY["selection"] == SELECTION

EXPORT_FREEZE_FINGERPRINT = fingerprint(
    EXPORT_BODY
)

EXPORT_FREEZE_PATH = (
    EXPORT_AUTH_DIR
    / "frozen_export_config.json"
)

EXPORT_FREEZE_PATH.write_text(

    json.dumps({

        "frozen_payload": EXPORT_BODY,

        "freeze_fingerprint": (
            EXPORT_FREEZE_FINGERPRINT
        ),

        "note": (
            "Export-only authorization for the "
            "unchanged original chronological "
            "CatBoost E5 checkpoint."
        )

    }, indent=2),

    encoding="utf-8"

)

EXPORT_APPROVAL = json.loads(
    FINAL_APPROVAL_PATH.read_text()
)

EXPORT_APPROVAL.update({

    "approved": True,

    "approved_by": APPROVED_BY,

    "approved_at_utc": EXPORT_TIMESTAMP,

    "authorized_scopes": [

        "final_execution",

        "export"

    ],

    "approved_freeze_fingerprint": (
        EXPORT_FREEZE_FINGERPRINT
    ),

    "note": (
        "Private backend research integration. "
        "No retraining, model reselection, "
        "chronological test rerun or midpoint release."
    )

})

EXPORT_APPROVAL_PATH = (
    EXPORT_AUTH_DIR
    / "approval_export.json"
)

EXPORT_APPROVAL_PATH.write_text(
    json.dumps(
        EXPORT_APPROVAL,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 4. Construct export-only configuration
# ============================================================

EXPORT_CONFIG = replace(

    FINAL_CONFIG,

    mode="final",

    approval_record=str(
        EXPORT_APPROVAL_PATH
    ),

    frozen_config=str(
        EXPORT_FREEZE_PATH
    ),

    resume_dir=None,

    run_final_test=False,

    run_grouped_outer=False,

    run_sensitivity_test=False,

    run_export=True

)

assert EXPORT_CONFIG.config_fingerprint == (
    CONFIG.config_fingerprint
)

assert not EXPORT_CONFIG.run_midpoint

require_approval(
    EXPORT_CONFIG,
    BUNDLE,
    "final_execution"
)

require_freeze(
    EXPORT_CONFIG,
    BUNDLE,
    "export"
)

# ============================================================
# 5. Execute official exporter
# ============================================================

print("\n" + "=" * 75)
print("EXPORTING FINAL CATBOOST E5 MODEL")
print("=" * 75)

export_start = perf_counter()

EXPORT_RUN_DIR = Path(
    run(EXPORT_CONFIG)
)

export_seconds = (
    perf_counter() - export_start
)

MODEL_ARTIFACT_DIR = (
    EXPORT_RUN_DIR
    / "models"
    / "selected_sleep_model"
)

REQUIRED_MODEL_FILES = [

    "model.joblib",

    "model_metadata.json",

    "feature_schema.json",

    "model_card.md"

]

for filename in REQUIRED_MODEL_FILES:

    assert (
        MODEL_ARTIFACT_DIR / filename
    ).is_file(), filename

export_metadata = json.loads(
    (
        MODEL_ARTIFACT_DIR
        / "model_metadata.json"
    ).read_text()
)

assert export_metadata["cpu_reload_verified"] is True

assert export_metadata["model_name"] == "catboost"

assert export_metadata["ordered_features"] == (
    SELECTION["ordered_features"]
)

assert not export_metadata["midpoint_validated"]

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256

print("\n" + "=" * 75)
print("OFFICIAL MODEL EXPORT COMPLETED")
print("=" * 75)

print("Model:", export_metadata["model_name"])

print(
    "Version:",
    export_metadata["model_version"]
)

print(
    "Features:",
    len(export_metadata["ordered_features"])
)

print("CPU reload verified: YES")
print("Original checkpoint changed: NO")
print("Model retrained: NO")
print("Test rerun: NO")

print(
    "Export runtime:",
    round(export_seconds, 3),
    "seconds"
)

print(
    "\nArtifact directory:",
    MODEL_ARTIFACT_DIR
)

print(
    "Delivery directory:",
    DELIVERY_ROOT
)


In [ ]:

# ============================================================
# AGENT 4 — BUILD FINAL DELIVERABLES
# ============================================================

import json
import shutil
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path
from importlib.metadata import version
from datetime import datetime, timezone

from ml.sleep.config import file_hash
from ml.sleep.logging_utils import safe_json


# ============================================================
# 1. Prepare delivery folders
# ============================================================

PACKAGE_DIR = (
    DELIVERY_ROOT
    / "agent4_final_model"
)

REPORT_DIR = (
    DELIVERY_ROOT
    / "agent4_final_reports"
)

PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def write_json(path, payload):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    path.write_text(

        json.dumps(
            safe_json(payload),
            indent=2,
            allow_nan=False
        ),

        encoding="utf-8"

    )


def read_json(path):

    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


# ============================================================
# 2. Load final metrics and comparison baselines
# ============================================================

validation_baselines = pd.read_csv(
    Path(DEV_RUN_DIR)
    / "metrics/baseline_metrics_validation.csv"
)

test_baselines = pd.read_csv(
    Path(FINAL_RUN_DIR)
    / "metrics/baseline_metrics_test.csv"
)

grouped_fold_results = pd.read_csv(
    Path(GROUPED_RUN_DIR)
    / "metrics/grouped_fold_results.csv"
)

assert len(grouped_fold_results) == 4

shap_importance = pd.read_csv(
    SHAP_DIR / "shap_global_importance.csv"
)

shap_groups = pd.read_csv(
    SHAP_DIR / "shap_group_importance.csv"
)

validation_b3 = float(

    validation_baselines.loc[

        validation_baselines["experiment_id"] == "B3",

        "mae_minutes"

    ].iloc[0]

)

test_b3 = float(

    test_baselines.loc[

        test_baselines["experiment_id"] == "B3",

        "mae_minutes"

    ].iloc[0]

)

test_mae = float(
    FINAL_TEST_METRICS["mae_minutes"]
)

validation_mae = float(
    SELECTION["mae_minutes"]
)

test_improvement_vs_b3 = (

    100

    * (test_b3 - test_mae)

    / test_b3

)

test_validation_change_pct = (

    100

    * (test_mae - validation_mae)

    / validation_mae

)

METRIC_KEYS = [

    "mae_minutes",

    "rmse_minutes",

    "participant_macro_mae_minutes",

    "mean_signed_error_minutes",

    "improvement_vs_persistence_pct",

    "persistence_mae_minutes",

    "n",

    "participants"

]


# ============================================================
# 3. Build unified evaluation summary
# ============================================================

EVALUATION_SUMMARY = {

    "project": "NASA Space Apps — Astronaut Health Monitoring",

    "agent": "Agent 4 — Sleep Prediction",

    "task": "next_main_sleep_tst_prediction",

    "model_name": "catboost",

    "model_version": export_metadata["model_version"],

    "feature_set": "E5",

    "n_features": 68,

    "target": "target_sleep_tst_minutes",

    "target_unit": "minutes",

    "prediction_horizon": "next_main_sleep",

    "model_selection": {

        "selection_metric": "chronological_validation_mae",

        "fit_policy": "train_only_no_validation_refit",

        "params": SELECTION["params"],

        "ordered_features": SELECTION["ordered_features"],

        "additional_experiments": [

            "Personalized residual learning",

            "Targeted feature ablation F1-F6",

            "Advanced CatBoost tuning F5/F6"

        ],

        "selected": "Original CatBoost E5 Tuned",

        "additional_experiments_selected": False,

        "test_used_for_selection": False

    },

    "data": {

        "source": "PMData",

        "preprocessing_version": BUNDLE.metadata["version"],

        "eligible_samples": BUNDLE.report["rows"],

        "participants": BUNDLE.report["participants"],

        "split_counts": BUNDLE.report["split_row_counts"],

        "fingerprints": BUNDLE.fingerprints

    },

    "chronological_validation": {

        "period": "February 2020",

        **{

            key: SELECTION.get(key)

            for key in METRIC_KEYS

        },

        "b3_mae_minutes": validation_b3,

        "improvement_vs_b3_pct": (

            100

            * (validation_b3 - validation_mae)

            / validation_b3

        ),

        "baselines": validation_baselines.to_dict(
            orient="records"
        )

    },

    "chronological_test": {

        "period": "March 2020",

        **{

            key: FINAL_TEST_METRICS.get(key)

            for key in METRIC_KEYS

        },

        "b3_mae_minutes": test_b3,

        "improvement_vs_b3_pct": (
            test_improvement_vs_b3
        ),

        "mae_change_vs_validation_pct": (
            test_validation_change_pct
        ),

        "baselines": test_baselines.to_dict(
            orient="records"
        ),

        "participant_cluster_bootstrap": (

            FINAL_TEST_METRICS.get(
                "participant_cluster_bootstrap"
            )

        )

    },

    "grouped_evaluation": {

        "completed": True,

        "outer_folds": 4,

        "inner_folds": 3,

        "protocol": GROUPED_RESULTS["protocol"],

        "metrics": GROUPED_RESULTS["metrics"],

        "participant_cluster_bootstrap": (

            GROUPED_RESULTS.get(
                "participant_cluster_bootstrap"
            )

        ),

        "interpretation": (
            "Evaluates the nested learning pipeline, "
            "not the identical frozen CatBoost checkpoint."
        )

    },

    "sensitivity_validation": {

        "completed": True,

        "protected_sensitivity_test_completed": False,

        "results": SENSITIVITY_RESULTS[

            [

                "scenario_id",

                "scenario_n",

                "matched_n",

                "mae_minutes",

                "rmse_minutes",

                "participant_macro_mae_minutes",

                "mae_change_vs_reference_pct",

                "target_changes",

                "changed_feature_values"

            ]

        ].to_dict(
            orient="records"
        ),

        "interpretation": (
            "Scenario-specific refits with unchanged "
            "frozen model family and hyperparameters."
        )

    },

    "explainability": {

        "completed": True,

        "method": SHAP_RESULTS["method"],

        "partition": "validation",

        "n_samples": SHAP_RESULTS["n_samples"],

        "additivity_verified": True,

        "max_additivity_error": (

            SHAP_RESULTS["max_additivity_error"]

        ),

        "top_features": (

            shap_importance.head(10)
            .to_dict(orient="records")

        ),

        "feature_groups": (

            shap_groups.to_dict(
                orient="records"
            )

        ),

        "runtime_local_explanations_available": False,

        "interpretation": (
            "Predictive attribution only, "
            "not evidence of causal effects."
        )

    },

    "release": {

        "scope": (
            "private_noncommercial_research_demo"
        ),

        "midpoint_validated": False,

        "midpoint_output": None,

        "individual_prediction_uncertainty": None,

        "clinical_validation": False,

        "astronaut_domain_validation": False,

        "freeze_fingerprint": FREEZE_FINGERPRINT,

        "source_checkpoint_sha256": (
            CHECKPOINT_SHA256
        ),

        "exported_model_sha256": file_hash(
            MODEL_ARTIFACT_DIR / "model.joblib"
        ),

        "cpu_reload_verified": True

    },

    "limitations": [

        "Small terrestrial wearable convenience cohort",

        "Correlated repeated observations per participant",

        "Wearable-estimated sleep duration",

        "Unknown participant timezone and ingestion timing",

        "Repeated development validation use",

        "Grouped pipeline evaluation differs from "
        "fixed-checkpoint new-user validation",

        "No paired bootstrap interval against B3",

        "No individual predictive uncertainty model",

        "PMData license discrepancy remains unresolved"

    ],

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat()

}

write_json(

    PACKAGE_DIR / "evaluation_summary.json",

    EVALUATION_SUMMARY

)

write_json(

    REPORT_DIR / "evaluation_summary.json",

    EVALUATION_SUMMARY

)


# ============================================================
# 4. Copy official model artifacts
# ============================================================

for filename in REQUIRED_MODEL_FILES:

    shutil.copy2(

        MODEL_ARTIFACT_DIR / filename,

        PACKAGE_DIR / filename

    )


# ============================================================
# 5. Copy standalone inference source
# ============================================================

SOURCE_MODULE_DIR = (
    Path(CODE_ROOT) / "ml/sleep"
)

DEST_MODULE_DIR = (
    PACKAGE_DIR / "ml/sleep"
)

DEST_MODULE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INFERENCE_MODULES = [

    "__init__.py",

    "models.py",

    "preprocessing.py",

    "inference.py"

]

for filename in INFERENCE_MODULES:

    source = SOURCE_MODULE_DIR / filename

    assert source.is_file()

    shutil.copy2(

        source,

        DEST_MODULE_DIR / filename

    )

# Do not include raw PMData or training checkpoints.


# ============================================================
# 6. Extend official deployment metadata
# ============================================================

metadata_path = (
    PACKAGE_DIR
    / "model_metadata.json"
)

PACKAGE_METADATA = read_json(
    metadata_path
)

PACKAGE_METADATA.update({

    "agent": "Agent 4",

    "task": "sleep_tst_prediction",

    "feature_set": "E5",

    "inference_device": "cpu",

    "prediction_unit": "minutes",

    "prediction_horizon": "next_main_sleep",

    "midpoint_validated": False,

    "uncertainty": None,

    "global_explainability_available": True,

    "explanation_available": False,

    "grouped_evaluation_completed": True,

    "sensitivity_validation_completed": True,

    "chronological_test_completed": True,

    "original_freeze_fingerprint": (
        FREEZE_FINGERPRINT
    ),

    "source_checkpoint_sha256": (
        CHECKPOINT_SHA256
    ),

    "exported_model_sha256": file_hash(
        PACKAGE_DIR / "model.joblib"
    ),

    "release_scope": (
        "private_noncommercial_research_demo"
    )

})

write_json(
    metadata_path,
    PACKAGE_METADATA
)


# ============================================================
# 7. Pin inference dependencies
# ============================================================

INFERENCE_DEPENDENCIES = [

    "numpy",

    "pandas",

    "scipy",

    "scikit-learn",

    "catboost",

    "joblib"

]

requirements = "\n".join(

    f"{name}=={version(name)}"

    for name in INFERENCE_DEPENDENCIES

)

(
    PACKAGE_DIR / "requirements.txt"
).write_text(

    requirements + "\n",

    encoding="utf-8"

)


# ============================================================
# 8. Include aggregate SHAP documentation
# ============================================================

EXPLAINABILITY_DIR = (
    PACKAGE_DIR / "explainability"
)

EXPLAINABILITY_DIR.mkdir(exist_ok=True)

for filename in [

    "shap_global_importance.csv",

    "shap_group_importance.csv",

    "shap_summary.json"

]:

    shutil.copy2(

        SHAP_DIR / filename,

        EXPLAINABILITY_DIR / filename

    )


# ============================================================
# 9. Build final model card
# ============================================================

model_card = f"""
# Agent 4 — Sleep Prediction Model Card

## Model identity

- Model: CatBoost Regressor
- Version: {PACKAGE_METADATA['model_version']}
- Feature configuration: E5
- Ordered predictors: 68
- Target: next main sleep total sleep time
- Unit: minutes
- Inference device: CPU
- Selected checkpoint: original development CatBoost E5
- Training policy: train only, no validation refit

## Intended use

Private research demonstration of personalized
sleep duration forecasting using historical wearable
and wellness information.

The model is intended to support nonclinical
situational awareness in the NASA Space Apps MVP.

It is not an astronaut-validated medical model.

## Dataset

PMData, a terrestrial wearable and wellness dataset.

Eligible observations: {BUNDLE.report['rows']}
Participants: {BUNDLE.report['participants']}

Chronological split:

- Training: 1057 observations, November–January
- Validation: 329 observations, February 2020
- Test: 296 observations, March 2020

## Model selection

The original CatBoost E5 tuned model was retained
after comparison with five model families,
six feature configurations, residual learning,
targeted feature ablation, and advanced tuning.

The protected chronological test was not used
to select the final model.

## Chronological performance

| Metric | Validation | Test |
|---|---:|---:|
| MAE (minutes) | {validation_mae:.4f} | {test_mae:.4f} |
| RMSE (minutes) | {SELECTION['rmse_minutes']:.4f} | {FINAL_TEST_METRICS['rmse_minutes']:.4f} |
| Participant-macro MAE | {SELECTION['participant_macro_mae_minutes']:.4f} | {FINAL_TEST_METRICS['participant_macro_mae_minutes']:.4f} |

Test improvement versus persistence:
{FINAL_TEST_METRICS['improvement_vs_persistence_pct']:.4f}%.

Test improvement versus personalized median:
{test_improvement_vs_b3:.4f}%.

Positive improvement means lower MAE.
A negative value means the model performed worse
than the corresponding baseline.

## Participant-independent evaluation

Nested 4-fold outer / 3-fold inner GroupKFold.

Pooled MAE:
{GROUPED_RESULTS['metrics']['mae_minutes']:.4f} minutes.

Pooled participant-macro MAE:
{GROUPED_RESULTS['metrics']['participant_macro_mae_minutes']:.4f} minutes.

Improvement versus persistence:
{GROUPED_RESULTS['metrics']['improvement_vs_persistence_pct']:.4f}%.

The grouped evaluation measures the learning
pipeline, not the identical frozen checkpoint.

## Sensitivity analysis

Three validation-only scenarios were evaluated:

- Wellness timestamp offset +1
- Wellness timestamp offset +2
- Wellness availability lag t-2

Each scenario uses the same selected model family,
features and fixed hyperparameters, with a separate
training-partition refit.

Detailed metrics are in evaluation_summary.json.

These scenarios do not recover actual timezones
or establish true ingestion timing.

## Explainability

Native CatBoost TreeSHAP on 329 validation samples.

SHAP additivity verified.

The most important global feature is:
{shap_importance.iloc[0]['feature']}.

Aggregate feature and group importance are provided
in the explainability directory.

SHAP explains model output, not causal relationships.

Per-request local explanations are not included.

## Input contract

Use feature_schema.json and the ordered feature
list in model_metadata.json.

Features must be prepared using the shared
point-in-time historical preprocessing contract.

The backend must validate previous sleep eligibility,
exact t-1 alignment, freshness, missingness,
and cutoff timestamps.

No raw wearable input is accepted directly.

## Output limitations

- Sleep TST output is a point prediction in minutes.
- Midpoint prediction is unavailable.
- Individual predictive uncertainty is unavailable.
- Input coverage is not model confidence.
- Do not infer bedtime, wake time or biological phase.
- Do not derive medical or flight-readiness clearance.

## Generalization and release restrictions

The source dataset is small, terrestrial and contains
correlated observations from repeated participants.

Wearable measurement error and unknown timezone
and ingestion remain important limitations.

No clinical or astronaut-domain validation exists.

The PMData repository/paper license discrepancy
remains unresolved. Keep this model private and
within the approved research-use scope pending
license clarification.

## Model integrity

Original checkpoint SHA256:
{CHECKPOINT_SHA256}

Exported model SHA256:
{PACKAGE_METADATA['exported_model_sha256']}

Chronological freeze fingerprint:
{FREEZE_FINGERPRINT}

CPU reload verification: passed.
"""

(
    PACKAGE_DIR / "model_card.md"
).write_text(

    model_card.strip() + "\n",

    encoding="utf-8"

)

(
    REPORT_DIR / "model_card.md"
).write_text(

    model_card.strip() + "\n",

    encoding="utf-8"

)


# ============================================================
# 10. Backend integration README
# ============================================================

backend_readme = """
# Agent 4 — Backend Integration

## Installation

Extract this ZIP into a private artifact directory.

Install compatible dependencies:

    pip install -r requirements.txt

The Python version and training environment are
recorded in model_metadata.json.

Keep the included ml/sleep package importable
through PYTHONPATH or the backend application root.

## Initialize once at application startup

    from ml.sleep.inference import SleepPredictor

    predictor = SleepPredictor(
        "artifacts/agent4_final_model"
    )

## Inference

    result = predictor.predict(feature_dict)

The feature dictionary must use the exact feature
names and historical definitions from
feature_schema.json.

The backend is responsible for creating features
from raw input and enforcing the preprocessing
and time-availability contract.

The inference helper does not reconstruct history.

## Synthetic software test

    python smoke_test.py

## Example input

    features = {
        "sleep_tst_prev": 420.0,
        "core_eligible": True,
        "cutoff_date": "2020-04-02",
        "sleep_prev_end": "2020-04-01T07:00:00",
        "cutoff_wallclock": "2020-04-02T20:00:00"
    }

This is a minimal synthetic software fixture.
It is not a complete real-world feature record.

For actual inference, provide all available
ordered predictors and correct observation counts,
freshness indicators, and missingness masks.

## Output fields

- predicted_sleep_tst_minutes
- predicted_sleep_midpoint_minutes
- sleep_tst_status
- sleep_midpoint_status
- model_version
- prediction_cutoff
- prediction_horizon
- input_coverage
- uncertainty
- explanation_available
- reason

TST status can be ok, insufficient_data, unavailable.

Null must not be converted into zero.

Midpoint and individual uncertainty are unavailable.

## Important for backend

- Load the model only once.
- No GPU is required.
- Do not change feature order.
- Do not retrain or replace model weights.
- Do not pass raw Fitbit data directly.
- Validate cutoff timestamps and exact t-1 history.
- Keep Agent 3 and Agent 4 prediction outputs separate.
- Input coverage is not confidence.
- Do not infer sleep quality, clinical diagnosis
  or readiness solely from predicted duration.

Global SHAP results are provided for documentation.

Runtime local SHAP explanations are not implemented.

## Security and release

Only deserialize the trusted model artifact
provided by the ML team.

Do not accept arbitrary user-supplied joblib files.

Use within the approved private research scope.

Do not publish model binaries or restricted
participant data without confirming license rights.
"""

(
    PACKAGE_DIR / "README.md"
).write_text(

    backend_readme.strip() + "\n",

    encoding="utf-8"

)


# ============================================================
# 11. Create CPU inference smoke test
# ============================================================

smoke_test = """
import json
from pathlib import Path

from ml.sleep.inference import SleepPredictor

ROOT = Path(__file__).resolve().parent

predictor = SleepPredictor(ROOT)

assert predictor.model.actual_device == "cpu"

assert len(predictor.model.columns) == 68

features = {
    "sleep_tst_prev": 420.0,
    "core_eligible": True,
    "cutoff_date": "2020-04-02",
    "sleep_prev_end": "2020-04-01T07:00:00",
    "cutoff_wallclock": "2020-04-02T20:00:00"
}

result = predictor.predict(features)

assert result["sleep_tst_status"] == "ok"

assert 0 < result["predicted_sleep_tst_minutes"] <= 1440

assert result["predicted_sleep_midpoint_minutes"] is None
assert result["sleep_midpoint_status"] == "unavailable"
assert result["uncertainty"] is None
assert result["explanation_available"] is False

missing = predictor.predict({})

assert missing["sleep_tst_status"] == "insufficient_data"
assert missing["predicted_sleep_tst_minutes"] is None

invalid_history = dict(features)

invalid_history["sleep_prev_end"] = "2020-03-30T07:00:00"

invalid = predictor.predict(invalid_history)

assert invalid["sleep_tst_status"] == "insufficient_data"

ineligible = dict(features)

ineligible["core_eligible"] = False

rejected = predictor.predict(ineligible)

assert rejected["sleep_tst_status"] == "insufficient_data"

print("CPU INFERENCE SMOKE TEST PASSED")

print(json.dumps(result, indent=2))
"""

(
    PACKAGE_DIR / "smoke_test.py"
).write_text(

    smoke_test.strip() + "\n",

    encoding="utf-8"

)


# ============================================================
# 12. Organize all important research reports
# ============================================================

COPIED_REPORTS = []
MISSING_REPORTS = []


def collect_reports(
    source_root,
    destination_name,
    required=None,
    optional=None,
    patterns=None
):

    source_root = Path(source_root)

    required = required or []

    optional = optional or []

    patterns = patterns or []

    destination = (
        REPORT_DIR
        / destination_name
    )

    destination.mkdir(
        parents=True,
        exist_ok=True
    )

    candidates = set(
        required + optional
    )

    for pattern in patterns:

        for path in source_root.glob(pattern):

            if path.is_file():

                candidates.add(
                    str(
                        path.relative_to(
                            source_root
                        )
                    )
                )

    for filename in sorted(candidates):

        source = source_root / filename

        if not source.is_file():

            if filename in required:

                raise FileNotFoundError(
                    f"Required report missing: {source}"
                )

            MISSING_REPORTS.append(
                str(source)
            )

            continue

        destination_path = (
            destination
            / filename
        )

        destination_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(
            source,
            destination_path
        )

        COPIED_REPORTS.append(

            str(
                destination_path.relative_to(
                    REPORT_DIR
                )
            )

        )


# ------------------------------------------------------------
# 01 — Original development
# ------------------------------------------------------------

collect_reports(

    DEV_RUN_DIR,

    "01_development",

    required=[

        "metrics/model_feature_results.csv",

        "metrics/baseline_metrics_validation.csv",

        "metrics/validation_permutation_importance.csv",

        "metadata/selection_proposal.json",

        "metadata/data_validation.json",

        "metadata/resume_identity.json",

        "metrics/runtime_summary.csv"

    ],

    optional=[

        "evaluation_report.md",

        "metadata/environment.json",

        "configs/run_config.json",

        "figures/model_feature_comparison.png",

        "figures/feature_ablation.png",

        "figures/validation_baselines.png",

        "figures/feature_missingness.png",

        "figures/fit_runtime.png"

    ],

    patterns=[

        "metrics/tuning_trials_*.csv",

        "figures/tuning_trials_*.png"

    ]

)


# ------------------------------------------------------------
# 02 — Frozen model and selection provenance
# ------------------------------------------------------------

collect_reports(

    FREEZE_DIR,

    "02_model_freeze",

    required=[

        "frozen_config.json",

        "final_execution_config.json",

        "final_test_completed.json"

    ]

)

# Keep actual authorization records private.
# Do not include them in the GitHub deliverable
# without a separate review.


# ------------------------------------------------------------
# 03 — Final chronological test
# ------------------------------------------------------------

collect_reports(

    FINAL_RUN_DIR,

    "03_chronological_test",

    required=[

        "metrics/final_test_metrics.json",

        "metrics/baseline_metrics_test.csv"

    ],

    optional=[

        "metrics/final_test_leaderboard.csv",

        "metrics/validation_vs_test.csv",

        "metrics/bootstrap_confidence_intervals.csv",

        "metrics/runtime_summary.csv",

        "metadata/data_validation.json",

        "evaluation_report.md",

        "figures/final_test_leaderboard.png"

    ]

)


# ------------------------------------------------------------
# 04 — Nested grouped evaluation
# ------------------------------------------------------------

collect_reports(

    GROUPED_RUN_DIR,

    "04_grouped_evaluation",

    required=[

        "metrics/grouped_fold_results.csv",

        "metrics/grouped_pooled_metrics.json"

    ],

    optional=[

        "metrics/runtime_summary.csv",

        "evaluation_report.md"

    ],

    patterns=[

        "metrics/outer_*_baseline_metrics_outer_heldout.csv",

        "metrics/tuning_trials_*.csv"

    ]

)


# ------------------------------------------------------------
# 05 — Sensitivity analysis
# ------------------------------------------------------------

collect_reports(

    SENSITIVITY_RUN_DIR,

    "05_sensitivity",

    required=[

        "metrics/sensitivity_validation_results.csv",

        "metrics/sensitivity_leaderboard.csv",

        "metrics/matched_reference_comparison.csv",

        "metrics/feature_change_summary.csv"

    ],

    optional=[

        "metrics/runtime_summary.csv",

        "metadata/sensitivity_protocol.json",

        "metadata/data_validation.json",

        "figures/sensitivity_mae_comparison.png",

        "figures/sensitivity_mae_change.png",

        "figures/sensitivity_missingness.png"

    ]

)

collect_reports(

    SENSITIVITY_PREFLIGHT_DIR,

    "05_sensitivity/preflight",

    required=[

        "scenario_summary.csv",

        "scenario_fingerprints.json"

    ]

)


# ------------------------------------------------------------
# 06 — Final SHAP explainability
# ------------------------------------------------------------

collect_reports(

    SHAP_DIR,

    "06_shap",

    required=[

        "shap_global_importance.csv",

        "shap_group_importance.csv",

        "shap_summary.json",

        "shap_direction_summary.csv"

    ],

    optional=[

        "shap_global_importance.png",

        "shap_group_importance.png",

        "shap_direction_summary.png"

    ]

)


# ------------------------------------------------------------
# 07 — Previous research experiments
# ------------------------------------------------------------

# Recover only aggregate result files from the previous
# experiment ZIPs or their saved run directories.

RESEARCH_ARCHIVES = {

    "residual_learning": {

        "zip": (
            "agent4_residual_learning_results.zip"
        ),

        "directory": "residual_learning",

        "files": [

            "experiment_log.csv",

            "inner_fold_metrics.csv",

            "alpha_search_results.csv",

            "residual_selection.json",

            "residual_validation_leaderboard.csv",

            "alpha_selection.png",

            "residual_comparison.png"

        ]

    },

    "targeted_ablation": {

        "zip": (
            "agent4_targeted_ablation_results.zip"
        ),

        "directory": "targeted_ablation",

        "files": [

            "feature_configurations.csv",

            "feature_lists.json",

            "inner_fold_metrics.csv",

            "inner_oof_leaderboard.csv",

            "internal_selection.json",

            "ablation_validation_leaderboard.csv",

            "stability_comparison.csv",

            "experiment_log.csv",

            "run_summary.json",

            "ablation_leaderboard.png",

            "monthly_stability.png",

            "feature_complexity.png"

        ]

    },

    "advanced_tuning": {

        "zip": (
            "agent4_advanced_tuning_results.zip"
        ),

        "directory": "advanced_tuning",

        "files": [

            "advanced_config.json",

            "feature_configurations.json",

            "fold_summary.csv",

            "all_optuna_trials.csv",

            "trials_T1.csv",

            "trials_T2.csv",

            "trials_T3.csv",

            "trials_T4.csv",

            "study_summary.csv",

            "internal_selection.json",

            "advanced_validation_leaderboard.csv",

            "monthly_stability.csv",

            "fit_runtime_log.csv",

            "run_summary.json",

            "optimization_history.png",

            "advanced_leaderboard.png",

            "advanced_monthly_stability.png"

        ]

    }

}

RESEARCH_RECOVERY = []

for name, spec in RESEARCH_ARCHIVES.items():

    destination = (

        REPORT_DIR

        / "07_research_archive"

        / name

    )

    destination.mkdir(
        parents=True,
        exist_ok=True
    )

    archive_path = (
        WORK_ROOT
        / spec["zip"]
    )

    copied = []

    if archive_path.is_file():

        with zipfile.ZipFile(
            archive_path, "r"
        ) as archive:

            for filename in spec["files"]:

                if filename in archive.namelist():

                    (
                        destination / filename
                    ).write_bytes(

                        archive.read(filename)

                    )

                    copied.append(filename)

        source_name = str(archive_path)

    else:

        source_root = (
            Path(DEV_RUN_DIR)
            / spec["directory"]
        )

        candidates = sorted(

            [

                path

                for path in source_root.glob("*")

                if path.is_dir()

            ]

        )

        source_name = None

        if candidates:

            source = candidates[-1]

            source_name = str(source)

            for filename in spec["files"]:

                path = source / filename

                if path.is_file():

                    shutil.copy2(

                        path,

                        destination / filename

                    )

                    copied.append(filename)

    RESEARCH_RECOVERY.append({

        "experiment": name,

        "source": source_name,

        "files_recovered": len(copied),

        "missing_files": [

            filename

            for filename in spec["files"]

            if filename not in copied

        ]

    })

write_json(

    REPORT_DIR

    / "07_research_archive"

    / "archive_recovery.json",

    RESEARCH_RECOVERY

)


# ------------------------------------------------------------
# 08 — Export evidence
# ------------------------------------------------------------

collect_reports(

    EXPORT_RUN_DIR,

    "08_export",

    required=[

        "metrics/runtime_summary.csv",

        "metadata/data_validation.json"

    ],

    optional=[

        "metadata/environment.json",

        "evaluation_report.md"

    ]

)

collect_reports(

    EXPORT_AUTH_DIR,

    "08_export/freeze",

    required=[

        "frozen_export_config.json"

    ]

)


# ============================================================
# 13. Generate consolidated final research report
# ============================================================

sensitivity_lines = []

for row in SENSITIVITY_RESULTS.itertuples():

    sensitivity_lines.append(

        f"| {row.scenario_id} | "

        f"{row.mae_minutes:.4f} | "

        f"{row.rmse_minutes:.4f} | "

        f"{row.mae_change_vs_reference_pct:.4f}% |"

    )

sensitivity_markdown = "\n".join(
    sensitivity_lines
)

top_shap_lines = []

for row in shap_importance.head(10).itertuples():

    top_shap_lines.append(

        f"| {row.rank} | "

        f"{row.feature} | "

        f"{row.mean_abs_shap_minutes:.4f} |"

    )

shap_markdown = "\n".join(
    top_shap_lines
)

final_report = f"""
# Agent 4 — Final Sleep Prediction Research Report

## Executive summary

Agent 4 developed and evaluated a personalized
next-main-sleep total sleep time regression model.

The selected model is the original tuned CatBoost
with full E5 feature configuration.

The final checkpoint was frozen before the protected
chronological test. Subsequent evaluations did not
change the selected checkpoint.

The optional midpoint task was intentionally skipped.

## Completed workflow

1. Data validation and preprocessing contract review
2. Baseline B0-B3 evaluation
3. Initial model-feature experiments
4. Hyperparameter optimization
5. Personalized residual learning
6. Targeted feature ablation
7. Advanced tuning F5/F6
8. Final model selection and freeze
9. Protected chronological test
10. Participant-independent nested grouped evaluation
11. Preprocessing sensitivity analysis
12. Native CatBoost TreeSHAP
13. CPU-compatible model export

## Model selection

Selected model: CatBoost E5 Tuned.

Validation MAE: {validation_mae:.4f} minutes.

Selected features: 68.

The residual, ablation and advanced tuning experiments
did not replace the original selected configuration.

## Chronological test results

| Metric | Validation | Test |
|---|---:|---:|
| MAE (min) | {validation_mae:.4f} | {test_mae:.4f} |
| RMSE (min) | {SELECTION['rmse_minutes']:.4f} | {FINAL_TEST_METRICS['rmse_minutes']:.4f} |
| Participant-macro MAE | {SELECTION['participant_macro_mae_minutes']:.4f} | {FINAL_TEST_METRICS['participant_macro_mae_minutes']:.4f} |

Test persistence improvement:
{FINAL_TEST_METRICS['improvement_vs_persistence_pct']:.4f}%.

Test personalized median MAE:
{test_b3:.4f} minutes.

Test improvement versus personalized median:
{test_improvement_vs_b3:.4f}%.

Test MAE change relative to validation:
{test_validation_change_pct:.4f}%.

Negative improvement means higher error than the baseline.

## Participant-independent nested evaluation

Outer folds: 4.

Inner folds: 3.

Pooled predictions:
{GROUPED_RESULTS['metrics']['n']}.

Pooled MAE:
{GROUPED_RESULTS['metrics']['mae_minutes']:.4f} minutes.

Pooled RMSE:
{GROUPED_RESULTS['metrics']['rmse_minutes']:.4f} minutes.

Pooled macro MAE:
{GROUPED_RESULTS['metrics']['participant_macro_mae_minutes']:.4f} minutes.

Improvement versus persistence:
{GROUPED_RESULTS['metrics']['improvement_vs_persistence_pct']:.4f}%.

The grouped evaluation assesses the nested learning
procedure, not the same frozen chronological checkpoint.

Secondary inner selection metrics for some tuned
candidates are placeholders; reported outer held-out
participant-macro metrics remain the actual evaluations.

## Preprocessing sensitivity analysis

| Scenario | MAE (min) | RMSE (min) | MAE change |
|---|---:|---:|---:|
{sensitivity_markdown}

The scenarios are validation-only comparisons with
separate training fits and fixed model configuration.

The results do not identify participants' true
timezones or actual ingestion delays.

## SHAP explainability

Method: {SHAP_RESULTS['method']}.

Partition: validation.

Samples: {SHAP_RESULTS['n_samples']}.

Additivity verification: passed.

| Rank | Feature | Mean absolute SHAP (min) |
|---:|---|---:|
{shap_markdown}

SHAP values measure contribution to model predictions,
not causal effects on sleep or health.

## Release status

- CPU-compatible model artifact: exported
- Midpoint prediction: unavailable
- Individual uncertainty: unavailable
- Runtime local SHAP: unavailable
- Offline global explainability: completed
- Release scope: private research demonstration

## Scientific and operational limitations

The model uses a small terrestrial convenience cohort
with wearable-estimated outcomes and correlated
participant-level observations.

Unknown timezone and data ingestion assumptions remain.

Chronological and internal validation periods were
reused during development.

The final model is not clinically validated and has
not been evaluated on astronaut-domain observations.

Grouped evaluation does not establish new-user
performance of the identical frozen checkpoint.

The small improvement over personalized median
should not be described as statistically significant
without a dedicated paired comparison.

PMData license discrepancy remains unresolved.

## Artifact integrity

Original freeze fingerprint:
{FREEZE_FINGERPRINT}

Original checkpoint SHA256:
{CHECKPOINT_SHA256}

Exported model SHA256:
{PACKAGE_METADATA['exported_model_sha256']}

See evaluation_summary.json for machine-readable
results and the detailed report subdirectories for
experiment-level evidence.
"""

(
    REPORT_DIR / "final_report.md"
).write_text(

    final_report.strip() + "\n",

    encoding="utf-8"

)

write_json(

    REPORT_DIR / "report_collection_summary.json",

    {

        "copied_reports": COPIED_REPORTS,

        "missing_optional_reports": MISSING_REPORTS,

        "research_archive_recovery": (
            RESEARCH_RECOVERY
        ),

        "contains_raw_pmdata": False,

        "contains_participant_predictions": False,

        "contains_model_checkpoints": False

    }

)

print("=" * 75)
print("FINAL MODEL AND REPORT DIRECTORIES CREATED")
print("=" * 75)

print("\nModel directory:", PACKAGE_DIR)

print("Report directory:", REPORT_DIR)

print(
    "\nValidation MAE:",
    round(validation_mae, 4)
)

print(
    "Test MAE:",
    round(test_mae, 4)
)

print(
    "Test improvement vs B3:",
    round(test_improvement_vs_b3, 4),
    "%"
)

print(
    "Grouped MAE:",
    round(
        GROUPED_RESULTS["metrics"]["mae_minutes"],
        4
    )
)

print("\nResearch archives recovered:")

display(
    pd.DataFrame(RESEARCH_RECOVERY)
)

print(
    "\nCore reports collected:",
    len(COPIED_REPORTS)
)

print(
    "Missing optional reports:",
    len(MISSING_REPORTS)
)

print("=" * 75)


In [ ]:

# ============================================================
# AGENT 4 — VERIFY AND DOWNLOAD FINAL DELIVERABLES
# ============================================================

import os
import sys
import json
import zipfile
import subprocess
import shutil

from pathlib import Path
from IPython.display import display, HTML

from ml.sleep.config import file_hash


# ============================================================
# 1. Verify backend package in a separate CPU process
# ============================================================

env = os.environ.copy()

env["CUDA_VISIBLE_DEVICES"] = ""

env["PYTHONPATH"] = str(
    PACKAGE_DIR.resolve()
)

verification = subprocess.run(

    [

        sys.executable,

        str(
            PACKAGE_DIR / "smoke_test.py"
        )

    ],

    cwd=str(PACKAGE_DIR),

    env=env,

    capture_output=True,

    text=True,

    timeout=120

)

print(verification.stdout)

if verification.returncode != 0:

    print(verification.stderr)

    raise RuntimeError(
        "Backend CPU smoke test failed. "
        "Do not distribute the model yet."
    )

assert (

    "CPU INFERENCE SMOKE TEST PASSED"

    in verification.stdout

)

CPU_VERIFICATION_PATH = (
    PACKAGE_DIR
    / "cpu_verification.txt"
)

CPU_VERIFICATION_PATH.write_text(

    verification.stdout,

    encoding="utf-8"

)

shutil.copy2(

    CPU_VERIFICATION_PATH,

    REPORT_DIR

    / "08_export"

    / "cpu_verification.txt"

)

assert file_hash(CHECKPOINT_PATH) == CHECKPOINT_SHA256

assert file_hash(
    PACKAGE_DIR / "model.joblib"
) == PACKAGE_METADATA["exported_model_sha256"]

print("\nCPU package verification: PASSED")


# ============================================================
# 2. Generate package manifests
# ============================================================

def build_manifest(root, package_type):

    root = Path(root)

    files = {}

    for path in sorted(root.rglob("*")):

        if not path.is_file():

            continue

        relative = path.relative_to(root)

        if "__pycache__" in relative.parts:

            continue

        if path.suffix == ".pyc":

            continue

        if path.name == "package_manifest.json":

            continue

        files[str(relative)] = {

            "sha256": file_hash(path),

            "size_bytes": path.stat().st_size

        }

    manifest = {

        "package_type": package_type,

        "agent": "Agent 4",

        "model": "CatBoost E5 Tuned",

        "model_version": (

            PACKAGE_METADATA["model_version"]

        ),

        "n_features": 68,

        "original_freeze_fingerprint": (
            FREEZE_FINGERPRINT
        ),

        "original_checkpoint_sha256": (
            CHECKPOINT_SHA256
        ),

        "exported_model_sha256": (

            PACKAGE_METADATA["exported_model_sha256"]

        ),

        "cpu_verified": True,

        "release_scope": (

            "private_noncommercial_research_demo"

        ),

        "contains_raw_data": False,

        "contains_participant_predictions": False,

        "files": files

    }

    write_json(

        root / "package_manifest.json",

        manifest

    )

    return manifest


MODEL_MANIFEST = build_manifest(

    PACKAGE_DIR,

    "backend_model"

)

REPORT_MANIFEST = build_manifest(

    REPORT_DIR,

    "research_reports"

)


# ============================================================
# 3. Create ZIP archives
# ============================================================

MODEL_ZIP = WORK_ROOT / "agent4_final_model.zip"

REPORT_ZIP = WORK_ROOT / "agent4_final_reports.zip"


def create_verified_zip(
    source_directory,
    zip_path,
    manifest
):

    source_directory = Path(
        source_directory
    )

    with zipfile.ZipFile(

        zip_path,

        mode="w",

        compression=zipfile.ZIP_DEFLATED

    ) as archive:

        for filename, details in (
            manifest["files"].items()
        ):

            path = (
                source_directory
                / filename
            )

            assert path.is_file()

            assert file_hash(path) == (
                details["sha256"]
            )

            archive.write(

                path,

                arcname=filename

            )

        archive.write(

            source_directory
            / "package_manifest.json",

            arcname="package_manifest.json"

        )

    # Verify ZIP integrity.

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as archive:

        assert archive.testzip() is None

        expected_files = (

            set(manifest["files"])

            | {"package_manifest.json"}

        )

        assert set(
            archive.namelist()
        ) == expected_files

        import hashlib

        for filename, details in (
            manifest["files"].items()
        ):

            digest = hashlib.sha256(

                archive.read(filename)

            ).hexdigest()

            assert digest == details["sha256"]

    return zip_path


create_verified_zip(

    PACKAGE_DIR,

    MODEL_ZIP,

    MODEL_MANIFEST

)

create_verified_zip(

    REPORT_DIR,

    REPORT_ZIP,

    REPORT_MANIFEST

)


# ============================================================
# 4. Verify report archive contains no model binaries
# ============================================================

with zipfile.ZipFile(
    REPORT_ZIP, "r"
) as archive:

    prohibited_suffixes = {

        ".joblib",

        ".pkl",

        ".pickle",

        ".parquet",

        ".db",

        ".sqlite",

        ".sqlite3"

    }

    for name in archive.namelist():

        assert Path(name).suffix.lower() not in (
            prohibited_suffixes
        ), name

        assert "private_checkpoints" not in name

        assert "predictions/" not in name


# ============================================================
# 5. Delivery summary
# ============================================================

DELIVERY_SUMMARY = {

    "status": "completed",

    "model_zip": str(MODEL_ZIP),

    "model_zip_sha256": file_hash(MODEL_ZIP),

    "model_zip_size_bytes": (

        MODEL_ZIP.stat().st_size

    ),

    "report_zip": str(REPORT_ZIP),

    "report_zip_sha256": file_hash(REPORT_ZIP),

    "report_zip_size_bytes": (

        REPORT_ZIP.stat().st_size

    ),

    "cpu_verified": True,

    "validation_mae_minutes": validation_mae,

    "test_mae_minutes": test_mae,

    "grouped_mae_minutes": (

        GROUPED_RESULTS["metrics"]["mae_minutes"]

    ),

    "midpoint_validated": False,

    "individual_uncertainty_available": False,

    "freeze_fingerprint": FREEZE_FINGERPRINT,

    "model_retrained": False,

    "test_rerun": False

}

write_json(

    DELIVERY_ROOT / "delivery_summary.json",

    DELIVERY_SUMMARY

)

print("\n" + "=" * 75)
print("AGENT 4 — FINAL DELIVERY COMPLETED")
print("=" * 75)

print("Selected model: Original CatBoost E5 Tuned")
print("Number of predictors: 68")

print(
    "Validation MAE:",
    round(validation_mae, 4)
)

print(
    "Test MAE:",
    round(test_mae, 4)
)

print(
    "Grouped MAE:",
    round(
        GROUPED_RESULTS["metrics"]["mae_minutes"],
        4
    )
)

print("\nCPU verification: PASSED")
print("Model ZIP integrity: PASSED")
print("Reports ZIP integrity: PASSED")

print("\nBackend package:")

print(
    MODEL_ZIP.name,
    round(
        MODEL_ZIP.stat().st_size / 1024**2,
        3
    ),
    "MB"
)

print("\nResearch reports:")

print(
    REPORT_ZIP.name,
    round(
        REPORT_ZIP.stat().st_size / 1024**2,
        3
    ),
    "MB"
)

print("\nModel unchanged: YES")
print("Ready for private backend integration: YES")

print("=" * 75)


# ============================================================
# Download the verified private files via Kaggle Output or local file explorer.
print(MODEL_ZIP)
print(REPORT_ZIP)
